# 실습 2 — ML Planner Open-loop·Closed-loop 평가

PLUTO와 Diffusion Planner를 동일한 nuPlan 시나리오에 적용하고, 모델 출력부터 공식 Open-loop·
Closed-loop 결과까지 하나의 흐름으로 분석한다. 먼저 모델 구조와 한 장면의 추론 결과를 확인한 뒤,
Open-loop에서는 로그 궤적 모사 성능을, Closed-loop에서는 계획을 실제로 반복 실행했을 때의
안전성과 주행 품질을 평가한다.

이 실습의 목적은 단순히 두 모델의 순위를 정하는 것이 아니다. 지표의 공식이 코드와 CSV의 값으로
어떻게 이어지는지 확인하고, 실패한 planning timestep을 영상에서 찾아 Open-loop 성능만으로는
설명할 수 없는 위험과 궤적 후처리의 필요성을 이해하는 것이 핵심이다.

## 학습 목표

1. Open-loop와 Closed-loop가 각각 무엇을 평가하며 어떤 조건을 고정해야 하는지 설명한다.
2. PLUTO의 입력·후보 궤적·confidence 구조를 확인하고 체크포인트로 한 장면을 추론한다.
3. Diffusion Planner의 입력과 반복 denoising 과정을 확인하고 한 장면의 궤적 생성을 시각화한다.
4. 동일한 시나리오에서 두 모델의 Open-loop 지표와 지평별 궤적 오차를 비교한다.
5. Closed-loop Non-reactive·Reactive의 차이와 곱셈 항·가중 항으로 구성된 지표 8개를 설명한다.
6. 진행률·속도 준수·Drivable area와 고전 TTC·nuPlan TTC 계산을 코드로 구현한다.
7. Closed-loop를 직접 실행하면서 planning timestep마다 8개 지표를 누적·시각화한다.
8. 두 모델의 Closed-loop CSV·영상·공식 점수를 분석하고 궤적 후처리가 필요한 이유를 설명한다.

## 실습 결과물

| 산출물 | 내용 |
|---|---|
| 한 장면 추론·시각화 | PLUTO 후보 궤적·점수와 Diffusion denoising 결과를 지도 위에서 확인한다 |
| Open-loop 비교 결과 | ADE·FDE·AHE·FHE·miss rate와 모델별 공식 점수·시나리오별 차이를 비교한다 |
| 개별 지표 구현 | 진행률·속도 준수·Drivable area·TTC 공식의 기호와 nuPlan 로직을 코드로 연결한다 |
| 실시간 주행 대시보드 | Closed-loop 영상 프레임과 8개 지표·timestep final을 함께 확인한다 |
| Closed-loop 결과 | Non-reactive·Reactive의 공식 점수와 곱셈 항 4개·가중 항 4개를 확인한다 |
| 실패 분석 자료 | CSV의 첫 점수 하락 시점과 영상의 차선 이탈·충돌 위험·추종 오차를 연결한다 |
| 최종 비교 | Open-loop와 Closed-loop가 서로 다른 질문임을 확인하고 후처리 요구사항을 정리한다 |

## 전체 실습 구성에서의 위치

```
 실습 0            실습 1               실습 2                실습 3
 환경 구성      →  nuPlan 프레임워크  →  ML planner 평가    →  MPC refinement
 데이터 배치        구성 요소와 교체      Open-loop·Closed-loop  궤적 후처리와 비교
```

실습 1에서 확인한 네 구성 요소 중 **planner만 PLUTO 또는 Diffusion으로 교체한다.** 평가 목적에
따라 controller와 observation preset을 명시적으로 고정해, 같은 조건 안에서만 모델을 비교한다.

## 진행 순서

| 절 | 내용 | 방식 |
|---|---|---|
| 0 | 준비 — 경로·환경변수·모델 확인 | |
| 1 | 무엇을 평가하는가 | 문서 |
| 2 | 샘플 시나리오 초기화 — 두 모델이 함께 쓴다 | API |
| 3 | **PLUTO** — 모델 구조 → 한 씬 추론 → Open-loop | API → config |
| 4 | **Diffusion Planner** — 모델 구조 → denoising 과정 → Open-loop | API → config |
| 5 | Open-loop 비교 — 두 모델 | config |
| 6 | Closed-loop 평가 구조 — Non-reactive·Reactive와 지표 8개 | 개념 |
| 7 | 진행률·속도 준수 | 빈칸 실습 |
| 8 | Drivable area·TTC — 개념식에서 nuPlan 구현까지 | 빈칸 → API |
| 9 | Closed-loop 실시간 지표 — 실행·누적·시각화 | API |
| 10 | PLUTO·Diffusion Closed-loop 실행과 실패 분석 | config → CSV·영상 |
| 11 | Open-loop·Closed-loop 대조와 궤적 후처리 필요성 | 비교 |
| 12 | 정리 | |

0~5절은 모델 구조와 Open-loop 성능을 다루고, 6~10절은 Closed-loop 지표를 직접 구현한 뒤 공식
시뮬레이션의 실패 지점을 추적한다. 11절에서 두 결과를 연결해 궤적 후처리의 필요성을 도출한다.

API로 객체를 직접 생성하는 셀은 모델·지표의 내부 흐름을 이해하기 위한 실습이고, Hydra 설정으로
전체 시나리오를 실행하는 셀에서 공식 집계 점수가 나온다. timestep 진단값과 공식 시나리오 점수를
같은 값으로 해석하지 않도록 각 절의 비교 단위를 확인한다.

- **커널은 `E2E Refinement`**여야 한다.
- 빈칸 셀은 `정답 형태` 주석을 참고해 TODO를 채운 뒤 검증 출력을 확인한다.
- MPC를 이용한 궤적 후처리는 실습 3에서 다룬다. 본 노트북은 `use_refinement=false`로 두므로
  **acados 를 설치하지 않았어도 진행할 수 있다.**
- 정식 시뮬레이션은 두 모델 × 세 프리셋(Open-loop·Closed-loop Non-reactive·Reactive)으로 총
  **여섯 번**이다. `run_sim`은 기존 결과가 있으면 재사용하므로 다시 열었을 때도 실행 셀을 순서대로
  통과하면 된다. 새로 실행하려면 해당 셀에서 `mode="rerun"`을 지정한다.


## 0. 준비

devkit 이 참조할 경로와 환경변수를 설정하고, 이후 절이 필요로 하는 체크포인트·GPU·데이터가
갖추어졌는지 확인한다.


In [ ]:
# 저장소 루트를 찾고 헬퍼를 로드한 뒤, devkit 이 참조할 환경변수를 설정한다.
import sys, pathlib

NB_DIR = pathlib.Path.cwd()
REPO_ROOT = NB_DIR if (NB_DIR / "run_simulation.py").exists() else NB_DIR.parent
assert (REPO_ROOT / "run_simulation.py").exists(), f"저장소 루트를 찾지 못했습니다: {NB_DIR}"

helper = next(REPO_ROOT.glob("practice/**/_practice2_helper.py"))
sys.path.insert(0, str(helper.parent))

# 헬퍼가 수정되면 커널이 캐시한 옛 모듈을 계속 쓴다. 이 셀을 다시 실행하면
# 최신 내용을 읽도록 reload 를 거친다.
import importlib
import _practice2_helper
importlib.reload(_practice2_helper)
from _practice2_helper import *          # noqa: F401,F403

REPO_ROOT = bootstrap()


In [ ]:
# 사전 점검 — 체크포인트 · GPU · 로그 DB · 위젯이 모두 준비되었는지 확인한다.
import importlib.util
import warnings

# PLUTO 의 attention mask 가 bool 이 아니라는 경고. 스텝마다 나와 출력을 덮으므로 끈다.
warnings.filterwarnings("ignore", message="Converting mask without torch.bool")

ckpt = REPO_ROOT / PLUTO_CKPT
dbs = list((REPO_ROOT / "data/db/test").glob("*.db"))

section("사전 점검")
print("python  :", sys.executable)
print("ckpt    :", ckpt.name, f"({ckpt.stat().st_size/1e6:.0f} MB)" if ckpt.exists() else "없음")
print("DB      :", len(dbs), "개")
print("widgets :", "OK" if importlib.util.find_spec("ipywidgets") else "없음")

import torch
print("torch   :", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      f"| {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "")

assert ckpt.exists(), f"PLUTO 체크포인트가 없습니다: {ckpt}"

# 개수를 고정하지 않고, 시나리오 필터가 요구하는 로그가 실제로 있는지 확인한다.
import yaml
need = yaml.safe_load(
    (REPO_ROOT / "config/scenario_filter/practice_scenarios.yaml").read_text())["log_names"]
absent = [n for n in need if not (REPO_ROOT / f"data/db/test/{n}.db").exists()]
assert not absent, f"필터가 요구하는 로그 DB 가 없습니다: {absent}"
print("\n✅ 준비 완료")


GPU 가 없어도 동작하지만 스텝당 추론이 크게 느려진다. 위 출력에서 `CUDA: False` 이면
3·4·10절의 정식 시뮬레이션 소요 시간이 몇 배로 늘어난다.


## 1. 무엇을 평가하는가

같은 planner 를 두 가지 방식으로 평가한다. 실습 1 §7 에서 규칙 기반 planner 로 확인한 그 구분이다.

| | Open-loop `open_loop_boxes` | Closed-loop `closed_loop_nonreactive_agents` |
|---|---|---|
| ego 의 움직임 | 로그를 그대로 재생 (`log_play_back_controller`) | 자신의 궤적으로 주행 (`two_stage_controller`) |
| 계획 궤적의 역할 | 채점 대상일 뿐, 실행되지 않는다 | 실행되어 다음 상태를 만든다 |
| 지표 | 로그 궤적과의 L2·헤딩 오차 (ADE/FDE) | 충돌·주행가능영역·진행률 등 8개 지표의 곱과 가중평균 |
| 답하는 질문 | 사람이 간 길과 얼마나 닮았는가 | 이 planner 로 주행하면 안전한가 |

두 지표는 성질이 달라 **서로 비교할 수 없다.** Open-loop 에서 오차가 작은 planner 가 Closed-loop 에서
0 점을 받을 수 있으며, 11절에서 실제로 그런 경우를 보게 된다.

평가 대상 planner 는 `RefinementPlanner` 이다. `use_refinement=false` 로 두면 **ML 궤적을
그대로** 내보내며, 매 스텝의 채점 결과를 프레임으로 그린다. 그 프레임을 모은 것이 산출물인
영상이다.


## 2. 샘플 시나리오 초기화

3·4 절에서 두 모델의 추론을 들여다볼 **한 장면**을 만든다. 실습 1 §2 와 같은 절차이므로
헬퍼 함수로 압축한다.

planner 생성자가 살아 있는 `AbstractScenario` 를 요구하므로(채점기를 생성자 안에서 만든다)
planner 보다 시나리오가 먼저 있어야 한다.

### 필터가 둘인 이유

| 쓰이는 곳 | 필터 | 왜 |
|---|---|---|
| 3·4 절 — 모델별 한 씬 추론 | `practice_single_scenario` | 모델이 **한 장면**에서 무엇을 하는지 보는 것이라 하나면 충분하다. 두 모델이 같은 장면을 봐야 비교가 된다 |
| 3·4·10절 — 정식 평가 | `practice_scenarios` | 점수는 시나리오마다 크게 갈리므로 **여러 개의 집계**라야 의미가 있다 |

아래 셀이 만드는 `scenario` 는 앞의 것이고, `run_sim` 이 돌리는 것은 뒤의 것이다.


In [ ]:
# 3·4 절이 들여다볼 한 장면을 만든다. 정식 평가(run_sim)는 practice_scenarios 를 따로 쓴다.
from nuplan.planning.utils.multithreading.worker_sequential import Sequential
from omegaconf import OmegaConf

SAMPLE_FILTER = "practice_single_scenario"
FILTER_YAML = OmegaConf.load(REPO_ROOT / f"config/scenario_filter/{SAMPLE_FILTER}.yaml")

builder = make_builder(REPO_ROOT)
scenarios = builder.get_scenarios(
    make_filter(scenario_tokens=list(FILTER_YAML.scenario_tokens),
                log_names=list(FILTER_YAML.log_names),
                timestamp_threshold_s=FILTER_YAML.timestamp_threshold_s),
    Sequential())

scenario = scenarios[0]
display(scenario_table(scenarios))
print("샘플:", scenario.token, "|", scenario.scenario_type)


## 3. PLUTO — 모델 구조 · 한 씬 추론 · Open-loop

<img src="assets/architecture_PLUTO_planner.png" width="880" alt="PLUTO 모델 구조">

<sub>출처: PLUTO 논문(*Push the Limit of Imitation Learning-based Planning for Autonomous Driving*) 원문 그림</sub>

PLUTO 는 장면을 토큰으로 부호화한 뒤 질의(query)로 궤적을 복원하는 Transformer 계열 모델이다.
좌측 인코더가 주변 차량(`E_A`) · 정적 객체(`E_O`) · 차선(`E_P`) · 자차(`E_AV`) 를 한 시퀀스로
통합하여 부호화하고, 우측 디코더가 횡방향·종방향 질의를 교차시켜 **여러 개의 후보 궤적과 점수**를
동시에 출력한다.

그림의 각 부분이 아래 생성자 인자에 대응한다.

| 그림 | 생성자 인자 |
|---|---|
| Transformer Encoder ×`L_enc` | `encoder_depth=4` |
| 디코더 블록 ×`L_dec` | `decoder_depth=4` |
| `Q_lat` × `Q_lon` 질의 격자 | `num_modes=12` |
| 토큰 차원 | `dim=128` |
| 출력 궤적 길이 | `future_steps=80` (0.1 s × 80 = 8 s) |
| 입력 이력 길이 | `history_steps=21` (과거 2 s) |

그림 오른쪽의 *Trajectory & Score* 는 후보가 여럿이라는 뜻이지만, 본 실습의 planner 가 읽는
것은 그중 **대표 궤적 하나**뿐이다. 화면에 보이는 점수는 그 궤적 하나를 채점한 값이다.

### 실행 구조

모델을 시뮬레이터에 장착하려면 네 개의 층이 필요하다. 각 층이 무엇을 책임지는지 확인하고 직접 만들어 본다.

```
 PlutoFeatureBuilder  →   PlanningModel   →  PlutoModelAdapter  →  RefinementPlanner
      모델 입력              추론 모델             모델 매니저              플래너
```

| 층 | 클래스 | 하는 일 |
|---|---|---|
| **모델 입력** | `PlutoFeatureBuilder` | 시뮬레이터 입력 → 모델 텐서 (반경 120 m, 주변 객체 48대, 과거 2 s) |
| **추론 모델** | `PlanningModel` | 텐서 → ego-local 궤적 `(80, 3)` |
| **모델 매니저** | `PlutoModelAdapter` | 모델마다 다른 forward 를 `AdapterOutput` 하나로 정규화하고 체크포인트를 소유 |
| **플래너** | `RefinementPlanner` | 궤적 선택 · 매 스텝 채점 · 렌더 · nuPlan `AbstractPlanner` 인터페이스 |

층이 나뉘어 있어 모델을 교체할 때 planner 를 고치지 않아도 된다. 4 절의 Diffusion Planner 는
어댑터만 바뀐다.

생성자 인자는 `config/planner/model_adapter/pluto.yaml` 의 값을 그대로 옮긴 것이며, 헬퍼의
`PLUTO_FEATURE_KWARGS` / `PLUTO_MODEL_KWARGS` 에 들어 있다.


In [ ]:
# PLUTO 를 네 층으로 직접 조립한다.
from src.feature_builders.pluto_feature_builder import PlutoFeatureBuilder
from src.models.pluto.pluto_model import PlanningModel
from src.planners.model_adapters import PlutoModelAdapter
from src.planners.refinement_planner import RefinementPlanner

SAMPLE_VIDEO_DIR = REPO_ROOT / "practice/videos/sample"

feature_builder = PlutoFeatureBuilder(**PLUTO_FEATURE_KWARGS)
model = PlanningModel(feature_builder=feature_builder, **PLUTO_MODEL_KWARGS)
adapter = PlutoModelAdapter(model, planner_ckpt=str(REPO_ROOT / PLUTO_CKPT))

planner = RefinementPlanner(
    model_adapter=adapter,
    scenario=scenario,
    use_refinement=False,               # 실습 2 는 MPC 를 쓰지 않는다
    driving_policy="ml",
    render=True,                        # 매 스텝 프레임을 모은다
    log_csv=False,                      # 커널에서는 CSV 가 누적된다 (아래 확인 사항 참조)
    render_mode="open",                 # 화면 하단을 Open-loop 지표로 그린다
    score_open_loop=True,               # 계획 궤적을 로그 ego 와 대조해 채점한다
    save_dir=str(SAMPLE_VIDEO_DIR),     # 기본값은 os.getcwd() 라 노트북 옆에 쌓인다
)
section("조립된 planner")
print("planner :", planner.name())
print("device  :", planner.device)
print("파라미터:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f} M")


### 체크포인트는 언제 로드되는가

생성자는 모델 구조만 만든다. **가중치는 `initialize()` 안에서 로드된다.** 어댑터의
`initialize` 가 체크포인트를 읽어 `load_state_dict` 하고 모델을 GPU 로 옮기기 때문이며,
이는 시뮬레이션이 시작될 때 한 번 일어난다.

파라미터 하나를 복사해 두고 `initialize()` 전후를 비교한다.


In [ ]:
# 생성 시점에는 가중치가 없다는 것을 확인한다 — initialize() 가 ckpt 를 로드한다.
import torch

key = [k for k in model.state_dict() if k.endswith("weight")][0]
before = model.state_dict()[key].clone()

# ego 를 로그 그대로 재생한다 — 이 절의 손 루프는 Open-loop 이다.
from nuplan.planning.simulation.controller.log_playback import LogPlaybackController

sim = make_sim(scenario, ego_controller=LogPlaybackController(scenario))
planner.initialize(sim.initialize())     # 여기서 load_state_dict + .to(device)

after = model.state_dict()[key]
section("체크포인트 로드 확인")
print("파라미터 :", key)
print("변화     :", not torch.equal(before, after.cpu()))
print("device   :", after.device)


### 같은 planner 를 Hydra 로 만들기

이 절 뒷부분의 정식 평가부터는 `run_simulation.py` 를 실행한다. 그때 planner 는 위와 같은 네 층으로 조립되지만,
인자는 yaml 에서 온다. `planner_builder` 가 `requires_scenario` 를 확인하고
`instantiate(config, scenario=scenario)` 로 시나리오를 주입하는 것이 위에서 `scenario=` 를
직접 넘긴 부분에 해당한다.


In [ ]:
# 위 네 줄과 같은 planner 를 Hydra 설정으로 만든다 — run_simulation.py 가 하는 방식이다.
# 프리셋은 planner 조립과 무관하므로 아무 것이나 넣어도 결과가 같다(여기서는 Closed-loop 프리셋).
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from nuplan.planning.script.builders.planner_builder import build_planners

GlobalHydra.instance().clear()            # 같은 프로세스에서 두 번 이상 부르려면 필요하다
with initialize_config_dir(config_dir=str(REPO_ROOT / "config")):
    cfg = compose(config_name="default_simulation", overrides=[
        "+simulation=closed_loop_nonreactive_agents",
        "planner=refinement_planner",
        "planner/model_adapter=pluto",
        "planner.refinement_planner.use_refinement=false",
        "scenario_builder=nuplan", "scenario_filter=practice_scenarios", "worker=sequential",
    ])

hydra_planner = build_planners(cfg.planner, scenario)[0]
print(type(hydra_planner).__name__, "| model_adapter:", type(hydra_planner._adapter).__name__)
del hydra_planner                          # 메모리에 두 벌을 남기지 않는다


### 빈칸 — 모델 출력을 지도 위로 옮기기

영상은 렌더러가 이미 변환을 마친 그림이다. 모델이 실제로 내놓는 것은
**ego 기준 상대좌표 `(T, 3)`** 이며, 지도 위에 그리려면 직접 옮겨야 한다.

$$
\begin{aligned}
x_g &= x_e + x_l\cos\theta - y_l\sin\theta \\
y_g &= y_e + x_l\sin\theta + y_l\cos\theta \\
\theta_g &= \mathrm{wrap}(\theta_l + \theta)
\end{aligned}
$$

회전은 **ego 의 heading $\theta$ 만큼 돌린 뒤** 위치를 더하는 순서이다. 순서를 바꾸면 안 된다.


In [ ]:
import numpy as np

def local_to_global(local, ego_x, ego_y, ego_h):
    """ego-local (T,3) [전방, 좌측, 상대헤딩] → global (T,3) [x, y, heading]."""
    local = np.asarray(local, dtype=float)
    c, s = np.cos(ego_h), np.sin(ego_h)

    # 예시 — 회전시킨 뒤 ego 위치를 더한다.
    x = ego_x + local[:, 0] * c - local[:, 1] * s

    # [TODO ①] 목적: 같은 회전을 y 성분에도 적용한다. 부호를 틀리면 궤적이
    #                지도 위에서 좌우로 뒤집혀 그려진다.
    #          채울 것: global y 좌표
    #          수식: y_g = y_e + x_l*sin(theta) + y_l*cos(theta)
    y = ...

    # [TODO ②] 목적: 각도는 더하는 것만으로 끝나지 않는다. 범위를 벗어나면
    #                이후 계산(헤딩 오차 등)이 2*pi 만큼 틀어진다.
    #          채울 것: global heading  (반드시 [-pi, pi) 로 되감을 것)
    #          수식: h_g = wrap(theta_l + theta) = arctan2(sin(theta_l+theta), cos(theta_l+theta))
    h = ...

    return np.stack([x, y, h], axis=1)


check_local_to_global(local_to_global)


채운 변환으로 **모델이 낸 점열을 직접** 그린다. 표와 영상은 점수만 보여 주지만,
여기서는 80 개 점이 실제로 어디에 놓이는지가 보인다.


In [ ]:
# 모델을 한 번 돌려 ego-local 궤적을 꺼내고, 위에서 채운 변환으로 지도 좌표에 올린다.
import matplotlib.pyplot as plt
import numpy as np

planner_input = sim.get_planner_input()
sample_ego = planner_input.history.current_state[0]

# 어댑터를 직접 부르기 전에 planner 를 한 번 통과시킨다. RefinementPlanner 가 ScenarioManager 에
# 현재 자차 상태를 넣어 주어야 feature builder 가 경로를 찾는다 — 건너뛰면 ego_state 가 None 이라
# route_roadblock_correction 에서 AttributeError 가 난다.
planner.compute_trajectory(planner_input)

out = adapter.build_and_forward(planner_input, planner._initialization)
ml_local = out.ml_local.cpu().numpy()          # (80, 3) — ego 기준 상대좌표

mine = local_to_global(ml_local, sample_ego.rear_axle.x, sample_ego.rear_axle.y, sample_ego.rear_axle.heading)
print("ml_local", ml_local.shape, "| 첫 점(local)", np.round(ml_local[0], 3))
print("변환 후 첫 점(global)", np.round(mine[0], 3))

log_xy = np.array([[s.rear_axle.x, s.rear_axle.y]
                   for s in scenario.get_expert_ego_trajectory()][:81])
plot_trajectory_points(ml_local, sample_ego.rear_axle.x, sample_ego.rear_axle.y,
                       sample_ego.rear_axle.heading, log_xy=log_xy)
plt.tight_layout(); plt.show()


### 후보 궤적과 점수 — PLUTO 의 모델 특성

위 셀이 그린 것은 궤적 **하나**다. 그런데 PLUTO 는 한 번의 forward 로 후보를 **여러 개**
내놓는다. 3 절 머리말 그림의 `Q_lat × Q_lon` 질의 격자가 그것이며, reference line 마다
`num_modes=12` 개의 종방향 후보가 붙는다.

planner 가 받는 것은 그중 **점수가 가장 높은 하나**뿐이다. 모델이 `probability` 를 평탄화해
`argmax` 로 고르고, 그것이 `output_trajectory` 가 되며, 어댑터가 그것만 `ml_local` 로 꺼낸다.
후보와 점수는 어댑터에서 버려진다 — 그래서 여기서는 **모델 forward 를 직접 부른다.**

Diffusion 의 denoising 과 대비되는 지점이다. 두 모델이 궤적 하나를 내놓기까지의 과정이
전혀 다른데도, planner 코드는 한 줄도 다르지 않다.

> `probability` 는 reference line 이 패딩된 자리를 `-1e6` 으로 채워 둔다. 거르지 않으면
> 유효한 후보들의 점수 차이가 색 하나로 뭉갠다.


In [ ]:
# 후보 궤적과 점수를 상위 10개만 꺼낸다.
import matplotlib.pyplot as plt
import numpy as np

candidates, scores, pluto_data = capture_pluto_candidates(
    adapter, planner_input, planner._initialization, top_k=10)

print("후보:", candidates.shape, "= (후보, 스텝, [x y yaw])")
print("점수:", np.round(scores, 3))
print("1위와 ml_local 의 최대 차이:",
      f"{np.abs(candidates[0] - ml_local).max():.1e}")

axes = plot_pluto_candidates(candidates, scores, data=pluto_data)
axes[0].figure.suptitle("PLUTO 후보 궤적 상위 10개 — 색과 겹침 순서가 모두 학습 점수", y=1.02)
plt.show()


**확인 사항**

- **1위 후보가 곧 `ml_local` 이다.** 위 출력의 최대 차이가 `0.0e+00` 이다. 화면에서 가장 진하고
  맨 위에 그려진 궤적 하나만 planner 로 넘어가고, 나머지 아홉 개는 버려진다.
- **이 장면(`traversing_pickup_dropoff`)은 모델이 헷갈리는 장면이다.** 열 개 후보의 점수가
  `-0.97` 부터 `-2.65` 사이에 몰려 있다 — 1·2위 차이가 0.33 뿐이다. 그런데 진행 거리는
  23 m · 2.8 m · 22.8 m … 로 크게 갈린다. **비슷한 점수의 후보가 전혀 다른 행동을 뜻하는**
  상황이며, 한 스텝 뒤에 순위가 뒤집히면 궤적이 그만큼 튄다.
- **두 패널이 모두 벌어진다.** 왼쪽에서는 후보들이 도로의 굽은 결을 따라 횡방향으로 7.6 m
  퍼지고, 오른쪽에서는 8 초 진행 거리가 2.8 m 부터 24.5 m 까지 세 무리로 갈린다.
  `Q_lat × Q_lon` 격자의 **두 축이 함께 작동하는** 장면이다.
- 직진 고속 주행 장면(`high_magnitude_speed`)에서는 반대로 왼쪽이 한 줄로 겹치고 오른쪽만
  벌어진다. `SAMPLE_FILTER` 의 토큰을 바꿔 가며 어느 패널이 벌어지는지 보면, 모델이 그
  장면에서 무엇을 고민하는지 읽을 수 있다.
- **업스트림 PLUTO 는 여기서 멈추지 않았다.** 후보들을 규칙 기반으로 다시 채점해 학습 점수와
  가중합한 뒤 고르고, 그 위에 비상 제동을 얹었다. 이 실습의 `RefinementPlanner` 에는 그 층이
  **일부러 없다** — 화면의 점수 차이를 후처리 탓으로 돌릴 수 없게 하기 위해서다.


### Open-loop 파이프라인을 직접 실행

Open-loop 31스텝을 직접 돌려, 영상의 한 프레임에 무엇이 담기는지 확인한다. 루프 본체는 실습 1 §5
의 세 줄과 같고, planner 와 `ego_controller` 가 바뀌었다.

`ego_controller` 가 `LogPlaybackController` 이므로 **자차는 planner 가 무엇을 내든 로그 그대로**
움직인다. `sim.propagate(trajectory)` 는 그대로 부르지만 controller 가 그 궤적을 쓰지 않는다 —
계획 궤적은 채점만 되고 실행되지 않는다. 아래 정식 Open-loop 평가와 같은 조건이다.

`compute_planner_trajectory` 가 아니라 `compute_trajectory` 를 부른다. 후자가 시뮬레이터가
실제로 호출하는 메서드이며, 스텝 소요 시간을 기록한다.

**31 스텝인 이유** — Open-loop 지표는 공식과 같이 **1 Hz 표본**만 누적한다. 루프는 10 Hz 로 도므로
`iteration % 10 == 0` 인 프레임, 즉 0 · 10 · 20 · 30 네 번만 누적된다. 20 스텝이면 두 번뿐이라
화면의 누적값이 거의 움직이지 않는다.

스텝당 2~3초가 걸린다 — 추론 + 채점 + 렌더가 모두 들어 있다. GPU가 없으면 훨씬 느리다.


In [ ]:
# Open-loop 31스텝을 직접 돌린다. 자차는 로그 재생이므로 아래 상태는 '로그의 자차' 이다.
import time

# 앞 절에서 한 씬을 추론할 때 iteration 0 이 이미 한 번 채점·렌더되었다. 지우고 시작해야
# 누적 표본이 격자 프레임(0·10·20·30) 네 개와 정확히 맞고, 이 셀을 다시 돌려도 결과가 같다.
planner._scorer.reset_open_loop()
planner._imgs.clear()

t0 = time.time()
for _ in range(31):
    planner_input = sim.get_planner_input()
    trajectory = planner.compute_trajectory(planner_input)
    sim.propagate(trajectory)
    sim_ego = sim.history.extract_ego_state[-1]
    print(f"iter {planner_input.iteration.index:3d}  "
          f"x={sim_ego.rear_axle.x:10.2f}  y={sim_ego.rear_axle.y:10.2f}  "
          f"v={sim_ego.dynamic_car_state.speed:5.2f} m/s  "
          f"a={sim_ego.dynamic_car_state.rear_axle_acceleration_2d.x:+5.2f} m/s²")
print(f"\n{time.time()-t0:.0f}s / 31스텝")


### 프레임 한 장 확인

`render=True` 이므로 planner 가 스텝마다 프레임을 모아 두었다. 그중 한 장을 크게 띄운다.


In [ ]:
# 모아 둔 프레임 중 한 장을 크게 띄운다 — 영상의 매 프레임이 이 화면이다.
import matplotlib.pyplot as plt

print("프레임:", len(planner._imgs), "장 |", planner._imgs[0].shape)
fig, ax = plt.subplots(figsize=(11, 11))
ax.imshow(planner._imgs[-1])
ax.axis("off")
plt.tight_layout()
plt.show()


화면 요소는 다음과 같다.

`render_mode="open"` 이므로 `OpenLoopSceneRender` 가 그린다. 지도·agent·궤적은
Closed-loop 렌더러와 같고, **점수 관련 두 자리만** Open-loop 지표로 바뀐다
([openloop_scene_render.py](../src/feature_builders/openloop_scene_render.py)).

| 위치 | 내용 |
|---|---|
| 좌하단 표 | Open-loop 지표 — ADE / FDE / AHE / FHE / miss rate 와 SCORE |
| 우하단 2패널 | 계획 궤적 80 스텝의 **변위 오차와 헤딩 오차** 곡선 |
| 자홍색 실선 | ML 이 계획한 궤적 |
| 주황 상자 | 자차. 파란 상자는 주변 차량이다 |
| 회색 실선 | 주변 객체의 **로그상 미래** |

Closed-loop 렌더러(10절)라면 이 두 자리에 충돌·주행가능영역·comfort 가 들어간다. 그 런에서
실제로 채점되는 지표를 화면에 올리는 것이라, 프리셋에 따라 화면이 달라진다.

### 영상으로 저장

모아 둔 프레임을 mp4 로 만든다. **손으로 돌린 루프에서는 이 호출을 직접 해야 한다.**
`run_simulation.py` 로 실행할 때는 `SimulationRunner` 가 시나리오 종료 시점에 불러 주지만,
여기에는 그 runner 가 없다. 한 번 부르면 모아 둔 프레임이 비워지므로 두 번 불러도 소용없다.


In [ ]:
# 모아 둔 프레임을 mp4 로 저장한다 — 손 루프에서는 이 호출을 직접 해야 한다.
planner.generate_planner_report()
saved = sorted(SAMPLE_VIDEO_DIR.glob("*.mp4"))
print(*[f"{p.name}  ({p.stat().st_size/1e6:.1f} MB)" for p in saved], sep="\n")


In [ ]:
# 저장된 mp4 를 노트북에서 재생한다.
from IPython.display import HTML

display(HTML(video_html(f"videos/sample/{saved[-1].name}")))


**확인 사항**

- **좌하단 표의 숫자는 표본 4 개짜리 중간값이다.** Open-loop 채점기는 공식과 같은 1 Hz 격자
  프레임(iteration 0 · 10 · 20 · 30)만 누적한다. 시나리오 전체를 채점한 공식 숫자는 바로 아래
  정식 평가에서 나온다.
- 이 절의 planner 는 `log_csv=False` 이므로 스텝별 채점 CSV 를 남기지 않는다. 아래 정식 런은
  `log_csv=true` 로 돌아 CSV 를 남기며, 그것을 시간축에 펼쳐 보는 것은 10절이다.


### Open-loop 정식 평가

`run_simulation.py` 로 `practice_scenarios` 전체를 Open-loop 평가한다. 공식 지표와 집계는 이 경로로만 나온다.

`+simulation=open_loop_boxes` 프리셋은 `planner` 그룹을 `log_future_planner` 로 덮어쓴다.
따라서 `planner=refinement_planner` 를 **프리셋 뒤에** 두어야 한다. 헬퍼의 `run_sim` 이 순서를
강제하며 실행 전 전체 명령을 그대로 출력한다.

### 이 프리셋이 구성하는 네 요소

실습 1 에서 확인한 네 축이 `open_loop_boxes` 에서 어떻게 채워지는지 확인한다.

| 구성 요소 | 선택된 옵션 | 의미 |
|---|---|---|
| planner | `refinement_planner` | 우리가 지정한다. 프리셋 기본값은 `log_future_planner` 이므로 뒤에서 덮어쓴다 |
| ego_controller | **`log_play_back_controller`** | 계획 궤적을 **사용하지 않고** 로그의 자차 상태를 그대로 되돌려준다 |
| observation | `box_observation` | 주변 물체를 로그에 기록된 대로 재생한다 (`TracksObservation`) |
| simulation_time_controller | `step_simulation_time_controller` | 시나리오의 iteration 을 0 부터 끝까지 한 칸씩 진행한다 |

핵심은 `ego_controller` 다. `log_play_back_controller` 는 planner 가 무엇을 내든 자차를 로그
그대로 놓으므로, **계획 궤적은 채점만 되고 실행되지 않는다.** 자차가 planner 를 따라가지 않으니
`driving_policy` 설정도 결과에 영향을 주지 않으며, 10절과 완전히 같은 planner 설정을 쓸 수 있다.

같은 시나리오를 4 절에서 Diffusion 으로 한 번 더 돌리며, 두 결과를 5 절에서 비교한다.

> 시나리오를 ray 워커 6개로 동시에 실행한다.


In [ ]:
# Open-loop 로 practice_scenarios 전체를 평가한다. 이미 결과가 있으면 건너뛴다
# (mode="rerun" 이면 다시 돈다). 위 한 씬 실습과 달리 여기서는 시나리오 여러 개가 필요하다.
OPEN = run_sim("open_loop_boxes", uid="practice2/pluto",
               scenario_filter="practice_scenarios", video_dir="videos")
print("결과:", OPEN)


### 최종 지표와 세부 지표

집계 parquet 에는 시나리오별 행, 시나리오 유형별 집계 행, 그리고 `final_score` 행이 섞여 있다.
`per_scenario_scores` 는 그중 시나리오별 행만 남긴다.


In [ ]:
# Open-loop 최종 점수와 시나리오별 세부 지표를 확인한다.
print(f"최종 점수: {final_score(OPEN):.4f}")
open_scores = per_scenario_scores(OPEN, columns=OPEN_BREAKDOWN)
display(open_scores.drop(columns=["video"]))


표의 다섯 개 지표는 임계로 정규화한 **점수**이다 — 앞 네 개는 `max(0, 1 - 오차/임계)` 연속값,
`miss_rate` 만 0/1 게이트이다(정의는 실습 1 §8).
실제로 몇 미터 틀렸는지는 `metrics/` 아래 원시 통계에만 있다. 3 · 5 · 8 초 지평에서의 값을 확인한다.

- **ADE** 평균 변위 오차, **FDE** 최종 변위 오차 (m) — 임계 8 m
- **AHE** 평균 헤딩 오차, **FHE** 최종 헤딩 오차 (rad) — 임계 0.8 rad
- **MR** 지평별 최대 오차가 허용치(6 · 8 · 16 m)를 넘은 표본 비율 — 0.3 을 넘으면 최종 점수가 0 이 되는 **곱셈 항**

최종 점수는 `MR 통과 여부 × (ADE + FDE + 2·AHE + 2·FHE) / 6` 이다. 헤딩 오차의 가중치가
거리 오차의 두 배이다.


In [ ]:
# 지평(3·5·8초)별 원시 오차값을 확인한다 — 집계표에는 없는 실제 오차(m, rad)이다.
horizon = openloop_horizon_table(OPEN)
display(horizon[[c for c in horizon.columns if "ADE" in c or "FDE" in c]])


### 시나리오별 영상 확인

왼쪽 목록에서 시나리오를 선택하면 해당 영상과 세부 지표가 나타난다. 점수가 낮은 순으로
정렬되어 있으므로 위쪽이 확인할 가치가 큰 장면이다.


In [ ]:
# 시나리오를 선택하면 그 시나리오의 영상과 지표가 나타난다.
display(scenario_video_browser(OPEN, tag="open_pluto", columns=OPEN_BREAKDOWN))


In [ ]:
# 실행 결과 점검 — 시나리오 수만큼 영상이 나왔는지 확인한다.
vids = collect_videos(OPEN, tag="open_pluto")
print(f"시나리오 {len(open_scores)}건 / 영상 {sum(v is not None for v in vids.values())}건")
display(runner_status(OPEN))
assert len(open_scores) > 0, "집계 parquet 이 비어 있습니다"


**확인 사항**

- 영상에서 자차(주황)와 로그 자차(회색)가 **끝까지 겹쳐 있다.** `log_play_back_controller` 가
  planner 의 궤적을 무시하고 로그를 재생하기 때문이다. 자홍색 계획 궤적만 로그에서 벗어난다.
- 화면의 점수표는 **Open-loop 지표**다. `run_sim` 이 Open-loop 런에 한해
  `score_open_loop=true` 와 `render_mode=open` 을 넣기 때문이며, 위의 손 루프에서 직접 준 인자와
  같은 것이다. 마지막 프레임의 누적 SCORE 는 위 집계표의 공식 점수와 일치한다.
- planner 를 무엇으로 바꾸어도 주행 자체는 같다. Open-loop 가 재는 것은 오직 궤적의 모양이다.


## 4. Diffusion Planner — 모델 구조 · denoising 과정 · Open-loop

<img src="assets/architecture_diffusion_planner.png" width="960" alt="Diffusion Planner 모델 구조">

<sub>출처: Diffusion Planner 논문 원문 그림</sub>

Diffusion Planner 는 궤적을 한 번에 내지 않는다. 잡음에서 시작해 **여러 번의 denoising 단계**를
거쳐 궤적을 복원하며(그림 우측 *Denoising Step*), 장면 정보는 조건으로 주입된다.

| 그림 | 하는 일 |
|---|---|
| Scenario Inputs (Neighbors · Lanes · Navigation · Static Obj) | 장면을 네 갈래로 나누어 입력한다 |
| Encoder (MLP-Mixer ×2 + MLP → Self-Attn ×N) | 네 갈래를 하나의 장면 조건으로 통합한다 |
| Diffusion Transformer | 장면 조건과 diffusion timestep 을 받아 한 단계 denoise 한다 |
| Denoising Step | 위 과정을 반복해 잡음을 궤적으로 만든다 |

PLUTO 와 구조가 전혀 다르지만 **planner 코드는 한 줄도 바뀌지 않는다.** 3 절의 네 층 중
어댑터가 모델별 forward 를 `AdapterOutput` 하나로 정규화하기 때문이며, 입력을 만드는 방식이
달라도(Diffusion 은 자체 `DataProcessor` 를 쓴다) 그 차이 또한 어댑터 안에 갇혀 있다.
바뀌는 것은 `planner/model_adapter=diffusion` 한 줄뿐이다.

이 절은 3 절과 **같은 순서**로 간다 — 모델 구조를 보고, 한 씬에 대한 입력과 추론 과정을 확인한 뒤,
같은 시나리오로 Open-loop 평가를 돌린다. 시나리오·프리셋·워커가 3 절과 모두 같으므로 5 절에서
두 결과를 그대로 비교할 수 있다.


### 한 씬 입력과 denoising 단계

3 절에서 PLUTO 는 한 번의 forward 로 궤적을 냈다. Diffusion 은 그렇지 않다 — **잡음에서
시작해 여러 단계를 거쳐** 궤적을 복원한다. 그 중간 단계를 직접 꺼내 본다.

3 절과 같은 장면(같은 시나리오의 iteration 0)을 쓴다. 다만 3 절의 `sim` 은 손 루프가 31 스텝
진행시켜 놓았으므로 새로 만든다 — `get_planner_input()` 이 돌려주는 이력은 **복사본이 아니라
살아 있는 버퍼**라서, 저장해 두어도 루프가 돌면 내용이 바뀐다.


In [ ]:
# Diffusion 을 3 절과 같은 네 층으로 조립한다. 바뀌는 것은 어댑터 한 층뿐이다.
from nuplan.planning.simulation.controller.log_playback import LogPlaybackController
from src.planners.model_adapters import DiffusionModelAdapter

DIFF_MODEL_DIR = REPO_ROOT / "data/model/diffusion_planner"

diff_adapter = DiffusionModelAdapter(
    planner_ckpt=str(DIFF_MODEL_DIR / "diffusion_planner.pth"),
    args_file=str(DIFF_MODEL_DIR / "diffusion_planner.json"),   # 구조 + 정규화 통계
    geometry_feature_builder=PlutoFeatureBuilder(**PLUTO_FEATURE_KWARGS),
    enable_ema=True,
)
diff_planner = RefinementPlanner(
    model_adapter=diff_adapter, scenario=scenario,
    use_refinement=False, driving_policy="ml", render=False, log_csv=False,
)

diff_sim = make_sim(scenario, ego_controller=LogPlaybackController(scenario))
diff_planner.initialize(diff_sim.initialize())
diff_input = diff_sim.get_planner_input()
diff_planner.compute_trajectory(diff_input)      # ScenarioManager 워밍업 — 3 절과 같은 이유

section("Diffusion planner")
print("어댑터   :", type(diff_adapter).__name__)
print("device   :", diff_planner.device)
print("파라미터 :", f"{sum(p.numel() for p in diff_adapter._planner.parameters()) / 1e6:.1f} M")


#### 모델이 보는 입력

위 그림의 *Scenario Inputs* 네 갈래가 그대로 텐서 네 개다. 어댑터는 이것을 만든 직후
`observation_normalizer` 를 통과시키므로 모델에 들어가는 값은 무차원이다. 그림으로 보려면
**정규화 전** 값이 필요해서 헬퍼가 같은 호출을 한 번 더 한다.

| 그림의 갈래 | 텐서 | 채널 |
|---|---|---|
| Neighbors | `neighbor_agents_past` (32, 21, 11) | x · y · cos · sin · vx · vy · 폭 · 길이 · 종류 onehot(3) |
| Static Obj | `static_objects` (5, 10) | x · y · cos · sin · 폭 · 길이 · 종류 onehot(4) |
| Lanes | `lanes` (70, 20, 12) | x · y · 방향(2) · 좌경계 offset(2) · 우경계 offset(2) · 신호 onehot(4) |
| Navigation | `route_lanes` (25, 20, 12) | `lanes` 와 같은 채널, 주행 경로에 속한 것만 |

**유효/패딩 마스크는 따로 없다.** 전 채널이 0 인 행이 패딩이며, 모델도 같은 규칙으로 마스크를
만든다. 좌표는 전부 **자차 뒷축 기준 미터**이고, 아래 denoising 중간 궤적과 같은 좌표계다.


In [ ]:
# Diffusion 이 실제로 보는 입력을 정규화 전 값으로 꺼내 그린다.
import matplotlib.pyplot as plt

scene = diffusion_scene_inputs(diff_adapter, diff_input)
for name, value in scene.items():
    print(f"{name:28s} {value.shape}")

fig, ax = plt.subplots(figsize=(11, 6))
plot_diffusion_scene(scene, ax=ax)
ax.set_title("Diffusion 이 보는 입력 씬 — ego-local, 정규화 전")
plt.show()


#### (심화) 중간 단계를 어떻게 꺼내는가

모델은 `dec["prediction"]` 하나만 돌려준다. 중간 `x_t` 는 밖으로 나오지 않는다. 그런데
샘플러는 이미 돌려줄 수 있게 되어 있다 — `DPM_Solver.sample(..., return_intermediate=True)`
가 단계마다 `x_t` 를 모아 주고, `dpm_sampler` 는 그 인자를 `sample_params` 로 그대로
통과시킨다. 호출부인 `decoder.py` 가 넘기지 않을 뿐이다.

그래서 **모델 코드를 고치지 않는다.** `decoder` 모듈에 `from ... import dpm_sampler` 로
바인딩된 그 **이름 하나만** 잠시 감싼다. 헬퍼의 `capture_denoising_steps` 가 하는 일이 그것이고,
지킬 것이 세 가지다.

1. 감싼 함수는 **`x0` 하나만** 돌려줘야 한다 — 호출부가 튜플을 받을 준비가 없다.
2. 모은 텐서는 **복사**해야 한다. 매 단계 현재 상태를 고정하는 `correcting_xt_fn` 이
   `x_t` 를 in-place 로 고치기 때문에, 복사하지 않으면 뒤 단계가 앞 단계를 덮어쓴다.
3. 모델이 내놓는 값은 정규화되어 있다. `decoder` 가 마지막에 하는 것과 똑같이
   `state_normalizer.inverse` 를 통과시켜야 미터가 된다.

`seed` 를 주면 `DIFFUSION_EVAL_SEED` 로 **초기 잡음이 고정**되어, 다시 돌려도 같은 그림이 나온다.
샘플러는 DPM-Solver++ 이고 단계 수는 10 으로 고정되어 있다. 여기에 워밍업과 마지막 정리 단계가
붙어 중간 텐서는 12 개가 된다.


In [ ]:
# denoising 중간 궤적을 모은다.
import numpy as np

steps, diff_out = capture_denoising_steps(
    diff_adapter, diff_input, diff_planner._initialization, seed=0)

print("중간 궤적:", steps.shape, "= (단계, 스텝, [x y yaw])")
print("마지막 단계와 AdapterOutput.ml_local 의 최대 차이:",
      f"{np.abs(steps[-1] - diff_out.ml_local.cpu().numpy()).max():.1e}")

fig, _ = plot_denoising_steps(steps, scene=scene)
fig.suptitle(f"한 씬에 대한 denoising {len(steps)} 단계", y=1.0, fontsize=12)
plt.show()


In [ ]:
# 같은 과정을 숫자로 본다 — 궤적이 언제 '궤적다워지는가'.
plot_denoising_convergence(steps)
plt.tight_layout()
plt.show()


**확인 사항**

- **마지막 단계가 곧 `AdapterOutput.ml_local` 이다.** 위 출력의 최대 차이가 `0.0e+00` 이다.
  planner 가 받는 궤적은 이 과정의 끝일 뿐, 중간 단계는 planner 에게 보이지 않는다.
- 초반 네댓 단계는 경로 길이가 1,300 m 를 넘는다. 8 초 동안 갈 수 있는 거리가 아니라
  **아직 잡음**이라는 뜻이다. 실제로 접히는 것은 5~8 단계에서 일어나고, 뒤 단계는 이미
  만들어진 궤적을 다듬는다 — 두 번째 그림의 이동량이 6 단계에서 정점을 찍고 급격히 준다.
- **잡음은 등방으로 퍼져 있는데 결과는 도로의 굽은 결을 따른다.** 8 단계부터 궤적이 휘기
  시작해 11 단계에서 차선의 곡률과 맞는다. 그 방향을 정하는 것이 `route_lanes` 이며, 위 입력
  씬 그림에서 진한 회색으로 그려진 것이 그것이다.
- 최종 경로 길이는 23.0 m 다. 같은 장면에서 PLUTO 의 1위 후보도 23.0 m 였다 — 전혀 다른
  방식으로 만든 궤적이 비슷한 결론에 도달했다. 다만 PLUTO 는 그 판단을 **후보 열 개 중
  하나를 고르는 방식**으로, Diffusion 은 **한 궤적을 열두 번 다듬는 방식**으로 했다.
- PLUTO 에는 이 과정이 없다. 한 번의 forward 로 후보 궤적들이 한꺼번에 나온다.
  **그런데 planner 코드는 두 모델에서 한 줄도 다르지 않다** — 어댑터가 이 차이를 전부
  `AdapterOutput` 안으로 감춘다.
- 이 절은 PLUTO 와 Diffusion 두 모델을 GPU 에 함께 올린다. 메모리가 모자라면 3 절의
  `planner` 와 `sim` 은 이후 절에서 쓰지 않으므로 `del` 로 정리해도 된다.


### Open-loop 정식 평가

3 절과 같은 프리셋·같은 시나리오로 돌린다. 바뀌는 것은 `planner/model_adapter=diffusion`
한 줄뿐이며, 헬퍼의 `run_sim(adapter="diffusion")` 이 그 override 를 넣는다.


In [ ]:
# Diffusion Planner 로 Open-loop 평가를 돌린다.
# 3 절의 PLUTO 런과 다른 인자는 adapter 와 uid 뿐이다 — 시나리오·프리셋·워커가 모두 같다.
DIFF_OPEN = run_sim("open_loop_boxes", uid="practice2/diffusion", adapter="diffusion",
                    scenario_filter="practice_scenarios", video_dir="videos")
print("결과:", DIFF_OPEN)


### 최종 지표와 세부 지표

PLUTO와 같은 순서로 공식 최종 점수, 시나리오별 세부 지표, 지평별 원시 오차와 영상을 확인한다.
여기서는 절차를 다시 설명하지 않고 3절과 **같은 열·같은 시나리오**가 나오는지만 확인한다.


In [ ]:
# Diffusion Open-loop 결과를 PLUTO와 같은 형식으로 읽는다.
print(f"최종 점수: {final_score(DIFF_OPEN):.4f}")
diff_open_scores = per_scenario_scores(DIFF_OPEN, columns=OPEN_BREAKDOWN)
display(diff_open_scores.drop(columns=["video"]))

diff_horizon = openloop_horizon_table(DIFF_OPEN)
display(diff_horizon[[c for c in diff_horizon.columns if "ADE" in c or "FDE" in c]])

display(scenario_video_browser(
    DIFF_OPEN, tag="open_diffusion", columns=OPEN_BREAKDOWN))
display(runner_status(DIFF_OPEN))


## 5. Open-loop 비교 — PLUTO 대 Diffusion

두 모델은 같은 `open_loop_boxes` 프리셋과 같은 `practice_scenarios`를 사용했다. 따라서 이 절에서는
모델만 다른 조건으로 시나리오별 점수와 세부 지표를 비교할 수 있다.

비교 순서는 다음과 같다.

1. 공통으로 성공한 시나리오의 총점을 나란히 놓는다.
2. ADE·FDE·AHE·FHE·miss rate 평균에서 차이가 난 항목을 찾는다.
3. 3·5·8초 원시 오차를 보고 차이가 어느 지평에서 커지는지 확인한다.

Open-loop 지표끼리는 비교할 수 있지만, 여기서 나온 순위를 Closed-loop 안전성의 순위로 해석하지 않는다.
그 질문은 실제 주행 결과가 준비된 11절에서 다시 확인한다.


In [ ]:
import pandas as pd

OPEN_RUNS = {"PLUTO": OPEN, "Diffusion": DIFF_OPEN}

display(compare_scenario_scores(OPEN_RUNS, columns=OPEN_BREAKDOWN))
open_breakdown = compare_breakdown(OPEN_RUNS, columns=OPEN_BREAKDOWN)
display(open_breakdown.round(3))
plot_breakdown_comparison(open_breakdown, title="Open-loop 세부 지표 — PLUTO 대 Diffusion")
plt.show()

horizon_compare = []
for model_name, run in OPEN_RUNS.items():
    table = openloop_horizon_table(run).copy()
    table.insert(0, "model", model_name)
    horizon_compare.append(table)
display(pd.concat(horizon_compare, ignore_index=True))


**확인 사항**

- 총점 차이가 어느 세부 지표에서 시작됐는지 먼저 확인한다.
- 짧은 지평은 비슷하지만 8초 지평에서 오차가 벌어지면 장기 계획의 차이로 해석한다.
- miss rate는 곱셈 gate이므로 소수의 실패 시나리오가 최종 점수를 크게 바꿀 수 있다.
- 점수가 크게 갈린 token은 두 모델의 Open-loop 영상에서 같은 장면을 선택해 궤적 모양을 비교한다.


## 6. Closed-loop 평가 구조

이 단원에서는 코드를 작성하기 전에 **무엇이 움직이고 무엇을 비교할 수 있는지** 먼저 정리한다.
Open-loop에서는 계획 궤적을 실행하지 않았지만, Closed-loop에서는 `two_stage_controller`가 계획을
추종한다. 실행된 ego 상태가 다음 planning timestep의 입력이 되므로 계획 오차와 추종 오차가 누적된다.

### 6.1 Non-reactive와 Reactive

두 실행에서 planner·ego controller·time controller는 같고, 주변 객체를 제공하는 **observation**만
달라진다. 이 실습의 `run_sim()`은 preset을 불러온 뒤 `planner=refinement_planner`를 적용하며,
`use_refinement=false`로 두 모델의 raw ML 계획을 비교한다.

| 항목 | Non-reactive | Reactive |
|---|---|---|
| preset | `closed_loop_nonreactive_agents` | `closed_loop_reactive_agents` |
| effective observation | `box_observation` | `idm_agents_observation` |
| 주변 차량 | 로그 궤적 재생 | IDM으로 새로 시뮬레이션 |
| ego에 대한 반응 | 없음 | 감속·정지·차간거리 유지 |
| 장점 | 같은 주변 거동이 반복되어 모델 비교가 쉽다 | ego와 주변 차량의 상호작용이 더 현실적이다 |
| 주의점 | 실제라면 피할 차량도 로그대로 움직여 충돌할 수 있다 | planner가 바뀌면 주변 차량의 움직임도 달라진다 |

**비교 규칙** — PLUTO와 Diffusion은 같은 preset 안에서 비교한다. Non-reactive와 Reactive의 점수
차이는 observation까지 함께 바뀐 결과이므로 모델 차이로 해석하지 않는다. 이후 분석은 재현성이 높은
Non-reactive를 중심으로 하고, Reactive는 별도 결과표로 확인한다.


### 6.2 Closed-loop 지표 8개

지표는 역할에 따라 두 묶음으로 읽는다.

- **곱셈 항**: 충돌·영역 이탈처럼 발생 자체가 치명적인 안전 gate
- **가중 항**: 진행·속도·승차감처럼 정도에 따라 감점되는 주행 품질

| 종류 | 지표 | 값/가중치 | 질문 |
|---|---|---:|---|
| 곱셈 | `no_ego_at_fault_collisions` | 0·0.5·1 | ego 과실 충돌이 없었는가 |
| 곱셈 | `drivable_area_compliance` | 0·1 | 차체가 주행 가능 영역 안에 있었는가 |
| 곱셈 | `driving_direction_compliance` | 0·0.5·1 | 차선 진행 방향을 지켰는가 |
| 곱셈 | `ego_is_making_progress` | 0·1 | 최소한의 경로 진행이 있었는가 |
| 가중 | `ego_progress_along_expert_route` | 5 | expert 대비 얼마나 진행했는가 |
| 가중 | `time_to_collision_within_bound` | 5 | 예상 충돌까지 충분한 시간이 남았는가 |
| 가중 | `speed_limit_compliance` | 4 | 제한속도 초과량과 초과 시간이 작은가 |
| 가중 | `ego_is_comfortable` | 2 | 가속도·jerk·yaw rate가 허용 범위인가 |

곱셈 항 하나가 0이면 최종 점수도 0이다. 가중 항은 가중치에 비례해 감점된다. 구체적인 집계식은
개별 지표 구현을 마친 뒤 9절에서 코드와 함께 확인한다. 7~8절에서 직접 구현하지 않는 나머지
4개 지표는 바로 다음 6.3절에서 nuPlan 원본 코드와 경계값으로 먼저 읽는다.

### 6.3 나머지 4개 지표 — nuPlan 원본 판정식을 코드로 읽기

7~8절에서는 progress·speed·Drivable area·TTC를 직접 구현한다. 여기서는 실시간 평가 함수에 함께
들어갈 나머지 네 지표의 **입력, 경계값, 반환값**을 확인한다. 설명 다음의 코드 셀은 nuPlan 원본의
핵심 분기와 같은 형태로 작성되어 있으므로 직접 값을 바꿔 실행할 수 있다.

#### 1) `no_ego_at_fault_collisions`

단순히 box가 닿았는지가 아니라 `classify_at_fault_collisions()`가 **ego가 피할 책임이 있었던 충돌**로
분류했는지를 본다. 객체 종류별 충돌 점수를 구한 뒤 VRU·차량·일반 객체 점수를 모두 곱한다.

공식 설정은 VRU·차량 허용 횟수 `0`, 일반 객체 허용 횟수 `1`이다. 따라서 VRU/차량 과실 충돌은 첫
1건부터 0점이고, 일반 객체는 `0건→1`, `1건→0.5`, `2건 이상→0`이다. 이 저장소의 planning timestep
진단 함수는 후보 필터링을 보수적으로 하기 위해 충돌이 있으면 바로 `0`, 없으면 `1`을 사용한다.


In [ ]:
# nuplan/.../no_ego_at_fault_collisions.py의 실제 component 함수를 호출한다.
import pandas as pd
from nuplan.planning.metrics.evaluation_metrics.common.no_ego_at_fault_collisions import (
    EgoAtFaultCollisionStatistics,
)

object_collision_cases = pd.DataFrame({
    "일반 객체 과실 충돌 횟수": [0, 1, 2],
    "component score": [
        EgoAtFaultCollisionStatistics._compute_collision_score(None, count, 1)
        for count in [0, 1, 2]
    ],
})
display(object_collision_cases)


#### 2) `driving_direction_compliance`

최근 1초 동안 lane 진행 방향의 반대로 이동한 최대 거리 `max_negative_progress`를 사용한다. 경계가
strict `<`이므로 정확히 `2 m`면 `0.5`, 정확히 `6 m`면 `0`이다.


In [ ]:
# nuplan/.../driving_direction_compliance.py의 핵심 분기를 그대로 실행한다.
def nuplan_direction_score(max_negative_progress):
    if max_negative_progress < 2.0:
        return 1.0
    elif max_negative_progress < 6.0:
        return 0.5
    else:
        return 0.0

reverse_distances = [0.0, 1.99, 2.0, 5.99, 6.0]
display(pd.DataFrame({
    "역방향 거리 [m]": reverse_distances,
    "direction score": [nuplan_direction_score(x) for x in reverse_distances],
}))


#### 3) `ego_is_making_progress`

별도 진행 거리를 다시 계산하지 않는다. 먼저 계산된 `ego_progress_along_expert_route`의 ratio가
`min_progress_threshold=0.2` 이상인지 검사한다. `>=`이므로 정확히 `0.2`도 통과한다.

이 항은 공식 시나리오 점수와 9~10절의 누적 점수에서 모두 곱셈 항으로 사용된다. 따라서 ratio가
`0.2`보다 작아 이 gate가 0이 되면, 나머지 가중 지표가 높아도 집계 점수는 0이다.


In [ ]:
# nuplan/.../ego_is_making_progress.py의 핵심 판정식
MIN_PROGRESS_RATIO = 0.20
progress_ratios = [0.19, 0.20, 0.70]
making_progress = [ratio >= MIN_PROGRESS_RATIO for ratio in progress_ratios]

display(pd.DataFrame({
    "progress ratio": progress_ratios,
    "making progress": making_progress,
}))


#### 4) `ego_is_comfortable`

하나의 평균값이 아니라 여섯 low-level metric의 `within_bound_status`가 **모두 True**인지 확인한다.

| low-level 신호 | 공식 bound |
|---|---:|
| longitudinal acceleration | `-4.05 ~ 2.40 m/s²` |
| absolute lateral acceleration | `≤ 4.89 m/s²` |
| absolute magnitude jerk | `≤ 8.37 m/s³` |
| absolute longitudinal jerk | `≤ 4.13 m/s³` |
| absolute yaw acceleration | `≤ 1.93 rad/s²` |
| absolute yaw rate | `≤ 0.95 rad/s` |


In [ ]:
# nuplan/.../ego_is_comfortable.py의 실제 AND 판정과 같은 형태다.
comfort_cases = pd.DataFrame({
    "six within_bound_status": [
        [True, True, True, True, True, True],
        [True, True, False, True, True, True],
    ],
})
comfort_cases["comfortable"] = comfort_cases["six within_bound_status"].map(
    lambda statuses: bool(np.all(statuses))
)
display(comfort_cases)


#### 네 판정 함수 한눈에 보기

앞의 표를 그래프로 바꾸면 네 지표가 모두 연속 감점이 아니라 **경계에서 값이 바뀌는 판정 함수**라는
점이 선명해진다. 충돌 그래프는 허용 횟수가 1인 일반 객체 component를 나타내며, VRU·차량은 첫
과실 충돌부터 0점이다.


In [ ]:
# 6.3에서 실행한 네 판정식을 2×2로 비교한다.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

collision_counts = np.arange(4)
collision_scores = [
    EgoAtFaultCollisionStatistics._compute_collision_score(None, int(count), 1)
    for count in collision_counts
]
axes[0, 0].stem(collision_counts, collision_scores, basefmt=" ")
axes[0, 0].set(title="At-fault collision: ordinary objects",
               xlabel="collision count", ylabel="component score",
               xticks=collision_counts, ylim=(-0.05, 1.05))

reverse_distance = np.linspace(0.0, 8.0, 161)
direction_scores = [nuplan_direction_score(value) for value in reverse_distance]
axes[0, 1].step(reverse_distance, direction_scores, where="post")
axes[0, 1].axvline(2.0, color="tab:orange", ls="--", label="2 m")
axes[0, 1].axvline(6.0, color="tab:red", ls="--", label="6 m")
axes[0, 1].set(title="Driving direction", xlabel="reverse distance [m]",
               ylabel="score", ylim=(-0.05, 1.05))
axes[0, 1].legend(fontsize=8)

ratio_grid = np.unique(np.r_[np.linspace(0.0, 1.0, 201), 0.2])
progress_gate = [float(ratio >= MIN_PROGRESS_RATIO) for ratio in ratio_grid]
axes[1, 0].step(ratio_grid, progress_gate, where="post", color="tab:green")
axes[1, 0].axvline(0.2, color="tab:red", ls="--", label="inclusive gate = 0.2")
axes[1, 0].scatter([0.2], [1.0], color="tab:green", zorder=3)
axes[1, 0].set(title="Ego is making progress", xlabel="progress ratio",
               ylabel="score", ylim=(-0.05, 1.05))
axes[1, 0].legend(fontsize=8)

failed_comfort_signals = np.arange(7)
comfort_scores = [float(failed == 0) for failed in failed_comfort_signals]
axes[1, 1].stem(failed_comfort_signals, comfort_scores, basefmt=" ")
axes[1, 1].set(title="Ego is comfortable", xlabel="failed low-level bounds",
               ylabel="score", xticks=failed_comfort_signals, ylim=(-0.05, 1.05))

for ax in axes.flat:
    ax.grid(alpha=.3)
fig.tight_layout(); plt.show()


## 7. 진행률과 속도 준수

먼저 지도 geometry나 주변 객체가 없어도 공식을 이해할 수 있는 두 연속 지표를 구현한다.
각 기호의 **뜻과 단위**를 확인한 뒤 바로 아래 빈칸을 채운다.

### 7.1 진행률 `ego_progress_along_expert_route`

| 기호·용어 | 의미 | 단위 |
|---|---|---:|
| $p_{ego}$ | 평가 구간의 시작·끝 ego 위치를 기준 경로에 투영해 얻은 전진 거리 | m |
| $p_{expert}$ | 같은 구간에서 로그 expert가 전진한 거리 | m |
| expert route | 로그 ego가 주행한 경로. 두 진행량을 재는 공통 기준선 | — |
| $\tau$ | 정지 구간의 0 나눗셈을 막는 하한. 공식값은 `2 m` | m |
| $\max(p,\tau)$ | 진행량이 작으면 계산에 최소 $\tau$를 사용한다는 뜻 | m |
| $\min(1,\cdot)$ | ego가 더 멀리 가도 점수를 1에서 포화한다는 뜻 | — |

$$
m_{progress}=\min\left(1,\frac{\max(p_{ego},\tau)}{\max(p_{expert},\tau)}\right),
\qquad \tau=2\,\mathrm m
$$

$p_{ego}< -\tau$이면 잡음 범위를 넘어 후진한 것이므로 바로 0이다. 연속값인 이 지표와 달리
`ego_is_making_progress`는 진행률이 최소 비율 `0.2` 이상인지 확인하는 곱셈 gate다.

아래 실습 셀에서는 표와 수식에서 조건을 찾아 구현한다. 주석은 필요한 개념만 힌트로 제공한다.

In [ ]:
def progress_score(ego_progress, expert_progress, threshold=2.0):
    '''expert 대비 진행률. threshold는 공식 score_progress_threshold [m].'''
    # [TODO ①] 허용 범위를 넘어 뒤로 진행한 경우를 판정한다.
    # 힌트: 전진은 양수, 후진은 음수다. threshold의 부호를 어디에 붙일지 생각한다.
    if ...:
        return 0.0

    # [TODO ②] 정지 구간을 위한 하한과 최대 점수 1을 함께 적용한다.
    # 힌트: 분자와 분모를 각각 먼저 보정한 뒤 ratio의 상한을 제한한다.
    ratio = ...
    return ratio


check_progress_score(progress_score)


### 7.2 제한속도 준수 `speed_limit_compliance`

| 기호·용어 | 의미 | 단위 |
|---|---|---:|
| $v_k$ | timestep $k$의 ego 속도 | m/s |
| $v_k^{limit}$ | 현재 lane/lane-connector의 제한속도 | m/s |
| $v_k^{over}$ | $\max(v_k-v_k^{limit},0)$, 제한속도를 넘은 양 | m/s |
| $\Delta t$ | 상태 sampling interval. 이 실습에서는 `0.1 s` | s |
| $T$ | 평가 상태열의 전체 지속시간 | s |
| $2.23\,m/s$ | 제한속도가 아니라 **초과량을 정규화하는 공식 기준값** | m/s |

$$
L_{speed}=\frac{\sum_k v_k^{over}\Delta t}{2.23\,T},\qquad
m_{speed}=\max(0,1-L_{speed})
$$

분자는 초과 속도 곡선 아래 면적이다. 따라서 잠깐 크게 초과한 경우와 오래 조금 초과한 경우가
같은 면적이면 같은 감점을 받는다.

In [ ]:
def speed_limit_score(overspeed, dt, duration, max_overspeed=2.23):
    '''overspeed는 timestep별 max(ego_speed - speed_limit, 0) [m/s].'''
    # [TODO ①] 초과 속도 곡선 아래 면적을 전체 시간과 기준 초과량으로 정규화한다.
    # 힌트: 일정 간격의 곡선 면적은 각 높이의 합에 sampling interval을 곱해 구한다.
    violation_loss = ...

    # [TODO ②] 위반 손실을 점수로 바꾸고 음수가 되지 않게 제한한다.
    # 힌트: 위반이 전혀 없을 때 점수는 1이고, 손실이 커질수록 감소해야 한다.
    score = ...
    return score


check_speed_limit_score(speed_limit_score)


### 7.3 연속 점수 함수 시각화

공식만 읽으면 threshold와 포화 구간을 놓치기 쉽다. 왼쪽은 expert 진행량을 고정했을 때 ego 진행량에
따라 progress 점수가 어떻게 변하는지, 오른쪽은 평가 구간의 평균 초과 속도가 speed 점수를 얼마나
감점하는지 보여준다. 앞 셀의 경계값 검증을 통과하면 그래프가 나타난다.


In [ ]:
import matplotlib.pyplot as plt

# 구현이 맞는지 경계값부터 확인한 뒤 그래프를 그린다.
try:
    progress_ready = (
        np.isclose(progress_score(5.0, 10.0), 0.5)
        and np.isclose(progress_score(12.0, 10.0), 1.0)
        and np.isclose(progress_score(-2.1, 10.0), 0.0)
    )
    speed_ready = (
        np.isclose(speed_limit_score(np.zeros(80), 0.1, 8.0), 1.0)
        and np.isclose(speed_limit_score(np.full(80, 2.23), 0.1, 8.0), 0.0)
    )
    if not (progress_ready and speed_ready):
        raise ValueError("7.1·7.2 구현의 경계값이 아직 맞지 않습니다.")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ego_progress_grid = np.linspace(-4.0, 14.0, 361)
    for expert_progress in [2.0, 5.0, 10.0]:
        scores = [progress_score(p, expert_progress) for p in ego_progress_grid]
        axes[0].plot(ego_progress_grid, scores,
                     label=f"expert progress = {expert_progress:g} m")
    axes[0].axvline(-2.0, color="tab:red", ls="--", label="reverse cutoff = -2 m")
    axes[0].axhline(0.2, color="tab:gray", ls=":", label="making-progress gate = 0.2")
    axes[0].set(xlabel="ego progress [m]", ylabel="progress score", ylim=(-0.05, 1.05),
                title="Progress: lower clamp, reverse cutoff, saturation")
    axes[0].grid(alpha=.3); axes[0].legend(fontsize=8)

    mean_overspeed = np.linspace(0.0, 4.46, 180)
    speed_scores = [
        speed_limit_score(np.full(80, value), dt=0.1, duration=8.0)
        for value in mean_overspeed
    ]
    axes[1].plot(mean_overspeed, speed_scores, color="tab:orange")
    axes[1].axvline(2.23, color="tab:red", ls="--", label="normalizer = 2.23 m/s")
    axes[1].fill_between(mean_overspeed, speed_scores, alpha=.15, color="tab:orange")
    axes[1].set(xlabel="mean overspeed over the window [m/s]", ylabel="speed score",
                ylim=(-0.05, 1.05), title="Speed: integrated overspeed penalty")
    axes[1].grid(alpha=.3); axes[1].legend(fontsize=8)

    fig.tight_layout(); plt.show()
except Exception as e:
    print(f"❌ 7.1·7.2 구현을 확인하십시오: {type(e).__name__}: {e}")


## 8. Geometry 기반 안전 지표 — Drivable area와 TTC

7절의 두 지표는 숫자 배열만으로 계산할 수 있었다. 이번 단원은 지도 polygon, ego footprint, 주변
객체 box가 필요하다. 학습 순서는 **개념용 작은 구현 → nuPlan 입력을 쓰는 구현**이다.

### 8.1 Drivable area map

nuPlan map은 한 장의 흑백 이미지가 아니라 차선과 교차로 영역을 polygon과 centerline으로 저장한
**semantic vector map**이다.

| 지도 요소 | 역할 |
|---|---|
| `LANE` | 일반 차선의 polygon·중심선·제한속도 |
| `LANE_CONNECTOR` | 교차로 안에서 진입 차선과 진출 차선을 연결하는 polygon |
| route lane | 현재 route에 포함된 lane·connector |
| `CROSSWALK` | 횡단보도 polygon |
| ego footprint | 길이·폭·heading을 가진 oriented box |

아래 그림은 임의의 도식이 아니라 현재 `scenario.map_api`에서 `sample_ego` 주변 50 m의 map object를
직접 조회한 결과다. 파란색 lane과 주황색 connector의 표면을 합치면 ego가 주행할 수 있는 도로 영역을
직관적으로 볼 수 있다.


In [ ]:
# 실제 nuPlan semantic vector map — sample ego 주변 50 m
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch, Polygon as MplPolygon
from nuplan.common.actor_state.state_representation import Point2D
from nuplan.common.maps.maps_datatypes import SemanticMapLayer

query = Point2D(sample_ego.center.x, sample_ego.center.y)
layers = [SemanticMapLayer.LANE, SemanticMapLayer.LANE_CONNECTOR, SemanticMapLayer.CROSSWALK]
nearby = scenario.map_api.get_proximal_map_objects(query, 50.0, layers)
route_ids = set(scenario.get_route_roadblock_ids())

fig, ax = plt.subplots(figsize=(9, 9))
styles = {
    SemanticMapLayer.LANE: dict(facecolor="#d9eaf7", edgecolor="#7aa6c2", alpha=.75),
    SemanticMapLayer.LANE_CONNECTOR: dict(facecolor="#ffe0b2", edgecolor="#d99032", alpha=.80),
    SemanticMapLayer.CROSSWALK: dict(facecolor="#eeeeee", edgecolor="#777777", alpha=.85),
}
for layer in layers:
    for obj in nearby.get(layer, []):
        polygon = np.asarray(obj.polygon.exterior.coords)
        style = dict(styles[layer])
        try:
            if obj.get_roadblock_id() in route_ids:
                style.update(facecolor="#8fc7ff", edgecolor="#1769aa", alpha=.9)
        except Exception:
            pass
        patch = MplPolygon(polygon[:, :2], closed=True, linewidth=.8, **style)
        if layer == SemanticMapLayer.CROSSWALK:
            patch.set_hatch("///")
        ax.add_patch(patch)
        if layer in (SemanticMapLayer.LANE, SemanticMapLayer.LANE_CONNECTOR):
            centerline = np.array([[p.x, p.y] for p in obj.baseline_path.discrete_path])
            ax.plot(centerline[:, 0], centerline[:, 1], "--", color="#555555",
                    lw=.7, alpha=.65, zorder=3)

footprint = np.asarray(sample_ego.car_footprint.geometry.exterior.coords)
ax.add_patch(MplPolygon(footprint[:, :2], closed=True, facecolor="#ff8c42",
                        edgecolor="black", linewidth=1.5, alpha=.95, zorder=8))
ax.scatter(footprint[:-1, 0], footprint[:-1, 1], s=28, c="red", zorder=9)
ax.arrow(sample_ego.center.x, sample_ego.center.y,
         6*np.cos(sample_ego.center.heading), 6*np.sin(sample_ego.center.heading),
         width=.12, head_width=1.1, color="black", length_includes_head=True, zorder=10)

cx, cy = sample_ego.center.x, sample_ego.center.y
ax.set(xlim=(cx-50, cx+50), ylim=(cy-50, cy+50),
       xlabel="global x [m]", ylabel="global y [m]")
ax.set_aspect("equal"); ax.grid(alpha=.15)
ax.set_title(f"nuPlan semantic vector map — {getattr(scenario.map_api, 'map_name', 'sample map')}")
ax.legend(handles=[
    Patch(facecolor="#d9eaf7", edgecolor="#7aa6c2", label="lane"),
    Patch(facecolor="#ffe0b2", edgecolor="#d99032", label="lane connector"),
    Patch(facecolor="#8fc7ff", edgecolor="#1769aa", label="route lane"),
    Patch(facecolor="#eeeeee", edgecolor="#777777", hatch="///", label="crosswalk"),
    Patch(facecolor="#ff8c42", edgecolor="black", label="ego footprint"),
], loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()


### 8.2 Drivable area 판정 로직

위 그림에서 주황색 ego box 전체가 도로 polygon 안에 있는지 확인한다고 생각하면 된다.

1. ego footprint에서 네 모서리를 구한다.
2. 각 모서리가 lane 또는 lane connector 안에 있는지 확인한다.
3. 밖에 있는 모서리가 있으면 가장 가까운 주행 가능 polygon까지의 거리를 잰다.
4. 그 거리가 `0.3 m` 이상이면 Drivable area 위반이다.
5. 위반은 한 번 발생하면 평가 window 끝까지 유지된다.

아래 개념 실습에서는 map query가 끝났다고 가정하고 `(timestep, corner)` 배열만 사용한다. 이 단원
끝에서는 실제 `map_api`와 `EgoState` sequence를 입력으로 같은 판정을 다시 구현한다.


In [ ]:
import numpy as np


def drivable_area_logic(outside, distance_to_drivable, tolerance=0.3):
    '''map query 결과 (T, 4)를 공식 누적 gate로 바꾼다.'''
    outside = np.asarray(outside, dtype=bool)
    distance = np.asarray(distance_to_drivable, dtype=float)
    assert outside.shape == distance.shape and outside.shape[1] == 4

    far_from_drivable_area = False
    per_step_outside = []
    for outside_t, distance_t in zip(outside, distance):
        per_step_outside.append(bool(np.any(outside_t)))

        # [TODO ①] 실제로 밖에 있는 모서리 중 허용 거리 이상 떨어진 것이 있는지 찾는다.
        # 힌트: 두 조건을 모서리별로 동시에 만족하는지 검사한 뒤 하나라도 있는지 확인한다.
        step_far = ...

        # [TODO ②] 이전 timestep의 위반을 잊지 않는 누적 상태를 만든다.
        # 힌트: 한 번 True가 된 뒤에는 다음 step이 정상이어도 False로 돌아가면 안 된다.
        far_from_drivable_area = ...

    # [TODO ③] 누적 위반 여부를 0/1 compliance 점수로 바꾼다.
    # 힌트: 위반이 없을 때만 1점이다.
    score = ...
    return score, per_step_outside


# 세 번째 timestep에 FR 모서리가 0.35 m 벗어나므로 최종 compliance가 낮아져야 한다.
outside_demo = [[0, 0, 0, 0], [0, 0, 0, 1], [0, 0, 0, 1]]
distance_demo = [[0, 0, 0, 0], [0, 0, 0, .12], [0, 0, 0, .35]]
try:
    demo_score, _ = drivable_area_logic(outside_demo, distance_demo)
    print("✅" if np.isclose(demo_score, 0.0) else "❌ TODO 확인", "demo score:", demo_score)
except Exception as e:
    print(f"❌ TODO를 채우십시오: {type(e).__name__}: {e}")


#### 실제 map 위 Drivable area gate 시각화

위에서 조회한 실제 nuPlan semantic map 위에서 ego footprint를 차선 바깥 방향으로 조금씩 이동시킨다.
왼쪽 그림은 차량이 도로 경계를 넘어갔다가 다시 들어오는 위치 변화를, 오른쪽 그림은 가장 멀리
벗어난 모서리의 거리와 누적 DAC 점수를 같은 timestep 축에 보여준다.

이 이동은 metric 동작을 분리해서 보기 위한 **synthetic lateral shift**이며 planner가 생성한 실제 궤적은
아니다. `0.3 m` 이상 이탈한 순간 점수가 `1→0`으로 바뀌고, 차량이 도로 안으로 돌아와도 평가
window의 누적 gate는 0으로 유지되는지 확인한다. 아래 긴 셀은 제공된 시각화 코드이므로 수정하지
않고, 왼쪽 차량 위치와 오른쪽 threshold crossing이 같은 timestep인지 관찰하면 된다.


In [ ]:
# 위에서 그린 실제 nuPlan map 위로 ego footprint를 이동시켜 DAC gate를 확인한다.
from shapely.affinity import translate
from shapely.geometry import Point
from shapely.ops import unary_union

# 경계값과 누적 gate를 먼저 점검한다.
def dac_score_for_single_corner(distance_m):
    outside = np.array([[False, False, False, True]])
    distances = np.array([[0.0, 0.0, 0.0, distance_m]])
    score, _ = drivable_area_logic(outside, distances, tolerance=0.3)
    return float(score)

try:
    dac_ready = (
        np.isclose(dac_score_for_single_corner(0.29), 1.0)
        and np.isclose(dac_score_for_single_corner(0.30), 0.0)
    )
    if not dac_ready:
        raise ValueError("8.2 구현의 threshold 또는 누적 gate가 아직 맞지 않습니다.")

    drivable_layers = [SemanticMapLayer.LANE, SemanticMapLayer.LANE_CONNECTOR]
    drivable_surface = unary_union([
        obj.polygon for layer in drivable_layers for obj in nearby.get(layer, [])
    ])
    base_footprint = sample_ego.car_footprint.geometry

    # ego heading의 왼쪽·오른쪽 중 가까운 도로 경계가 있는 방향을 자동으로 선택한다.
    lateral = np.array([-np.sin(sample_ego.center.heading), np.cos(sample_ego.center.heading)])
    candidate_offsets = np.linspace(0.0, 12.0, 97)

    def shifted_footprint_query(offset_m, direction):
        dx, dy = direction * offset_m * lateral
        footprint = translate(base_footprint, xoff=dx, yoff=dy)
        corners = np.asarray(footprint.exterior.coords)[:-1, :2]
        points = [Point(float(x), float(y)) for x, y in corners]
        outside = np.array([not drivable_surface.covers(point) for point in points])
        distances = np.array([point.distance(drivable_surface) for point in points])
        return footprint, outside, distances

    def first_offset_reaching(direction, target=0.6):
        for offset in candidate_offsets:
            _, _, distances = shifted_footprint_query(offset, direction)
            if np.max(distances) >= target:
                return float(offset)
        return float(candidate_offsets[-1])

    direction = min((-1.0, 1.0), key=first_offset_reaching)
    peak_offset = first_offset_reaching(direction)
    outbound_offsets = np.linspace(0.0, peak_offset, 8)
    offsets = np.concatenate([outbound_offsets, outbound_offsets[-2::-1]])

    shifted_footprints, outside_rows, distance_rows = [], [], []
    for offset in offsets:
        footprint, outside, distances = shifted_footprint_query(offset, direction)
        shifted_footprints.append(footprint)
        outside_rows.append(outside)
        distance_rows.append(distances)

    outside_sequence = np.asarray(outside_rows, dtype=bool)
    distance_sequence = np.asarray(distance_rows, dtype=float)
    max_corner_distance = distance_sequence.max(axis=1)
    prefix_scores = np.array([
        drivable_area_logic(outside_sequence[:i], distance_sequence[:i], tolerance=0.3)[0]
        for i in range(1, len(offsets) + 1)
    ])
    violation_steps = np.flatnonzero(max_corner_distance >= 0.3)
    if len(violation_steps) == 0:
        raise RuntimeError("현재 map에서 0.3 m 이탈 pose를 만들지 못했습니다.")
    violation_step = int(violation_steps[0])
    peak_step = int(np.argmax(offsets))

    fig, (ax_map, ax_metric) = plt.subplots(
        1, 2, figsize=(15, 6.2), layout="constrained")

    # 8.1절과 같은 실제 semantic map을 다시 그린다. 좌표는 초기 ego 기준으로 표시한다.
    map_origin = np.array([sample_ego.center.x, sample_ego.center.y])
    for layer in drivable_layers:
        for obj in nearby.get(layer, []):
            polygon = np.asarray(obj.polygon.exterior.coords)
            style = dict(styles[layer])
            try:
                if obj.get_roadblock_id() in route_ids:
                    style.update(facecolor="#8fc7ff", edgecolor="#1769aa", alpha=.9)
            except Exception:
                pass
            ax_map.add_patch(MplPolygon(
                polygon[:, :2] - map_origin, closed=True, linewidth=.8, zorder=1, **style))

    centers = np.array([
        direction * offset * lateral
        for offset in offsets
    ])
    ax_map.plot(centers[:, 0], centers[:, 1], "k.--", lw=1.2, ms=4,
                label="synthetic lateral motion", zorder=5)

    snapshot_steps = sorted(set([
        0, max(0, violation_step - 1), violation_step, peak_step, len(offsets) - 1
    ]))
    for step in snapshot_steps:
        xy = np.asarray(shifted_footprints[step].exterior.coords)
        returned = step == len(offsets) - 1
        passed = prefix_scores[step] > 0.5
        facecolor = "none" if returned else ("#55a868" if passed else "#c44e52")
        edgecolor = "#8172b3" if returned else ("#236b3b" if passed else "#8b1a1a")
        linestyle = "--" if returned else "-"
        ax_map.add_patch(MplPolygon(
            xy[:, :2] - map_origin, closed=True, facecolor=facecolor, edgecolor=edgecolor,
            linewidth=2.0, linestyle=linestyle, alpha=.82, zorder=8))
        label = f"t={step}, DAC={prefix_scores[step]:.0f}"
        if returned:
            label += " (re-entry)"
        if returned:
            label_offset = (25, 35)
        elif step == 0:
            label_offset = (-105, 35)
        elif step == violation_step:
            label_offset = (15, -42)
        else:
            label_offset = (-105, -30)
        ax_map.annotate(
            label, centers[step], xytext=label_offset, textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color=edgecolor, lw=.8),
            fontsize=8, color=edgecolor, weight="bold", zorder=10)

    all_xy = np.vstack([
        np.asarray(p.exterior.coords)[:, :2] - map_origin for p in shifted_footprints
    ])
    pad = 7.0
    ax_map.set(xlim=(all_xy[:, 0].min() - pad, all_xy[:, 0].max() + pad),
               ylim=(all_xy[:, 1].min() - pad, all_xy[:, 1].max() + pad),
               xlabel="x from initial ego [m]", ylabel="y from initial ego [m]",
               title="Ego leaves and re-enters the drivable area")
    ax_map.set_aspect("equal"); ax_map.grid(alpha=.15)
    ax_map.legend(loc="best", fontsize=8)

    steps = np.arange(len(offsets))
    ax_metric.plot(steps, max_corner_distance, "o-", color="tab:blue",
                   label="max outside-corner distance")
    ax_metric.axhline(0.3, color="tab:red", ls="--", label="violation threshold = 0.3 m")
    ax_metric.axvline(violation_step, color="tab:red", ls=":", alpha=.8)
    ax_metric.axvspan(peak_step, steps[-1], color="tab:purple", alpha=.08,
                      label="vehicle returns toward road")
    score_ax = ax_metric.twinx()
    score_ax.step(steps, prefix_scores, where="post", color="tab:orange", lw=2.4,
                  label="cumulative DAC score")
    score_ax.scatter([steps[-1]], [prefix_scores[-1]], color="tab:purple", zorder=5)
    score_ax.annotate("re-entered, but score stays 0",
                      (steps[-1], prefix_scores[-1]), xytext=(-145, 28),
                      textcoords="offset points", arrowprops=dict(arrowstyle="->"), fontsize=8)

    ax_metric.set(xlabel="timestep", ylabel="max outside distance [m]",
                  title="Threshold crossing and latched score")
    score_ax.set(ylabel="DAC score", ylim=(-0.05, 1.05))
    score_ax.set_zorder(1)
    ax_metric.set_zorder(2); ax_metric.patch.set_visible(False)
    lines1, labels1 = ax_metric.get_legend_handles_labels()
    lines2, labels2 = score_ax.get_legend_handles_labels()
    ax_metric.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")
    ax_metric.grid(alpha=.3)

    plt.show()
except Exception as e:
    print(f"❌ 8.2 구현을 확인하십시오: {type(e).__name__}: {e}")


### 8.3 고전적인 TTC부터 구현하기

먼저 같은 차선에서 앞차를 따라가는 가장 단순한 상황을 생각한다.

- $d$: ego 앞면과 앞차 뒷면 사이의 남은 거리 `[m]`
- $v_e$: ego 속도 `[m/s]`
- $v_a$: 앞차 속도 `[m/s]`
- $v_e-v_a$: 두 차량 사이가 줄어드는 속도, 즉 closing speed `[m/s]`

$$
TTC=\begin{cases}
\dfrac{d}{v_e-v_a}, & v_e-v_a>0\\
\infty, & v_e-v_a\le0
\end{cases}
$$

예를 들어 gap이 `20 m`이고 ego가 `10 m/s`, 앞차가 `5 m/s`이면 매초 5 m씩 가까워지므로
`TTC=20/5=4 s`다. ego가 더 느리거나 같은 속도라면 거리가 줄지 않으므로 TTC는 무한대다.

이 식은 이해하기 쉽지만 차량을 점으로 보고, 같은 직선 위의 앞차 하나만 다룬다. 그래서 차체의
길이·폭이나 교차로에서 옆으로 들어오는 차량을 표현하기 어렵다. 아래 빈칸에서 1차원 공식을 먼저
구현한 뒤, 8.4절에서 **점을 oriented box로 바꾸고 0.1초씩 미래로 이동시키는 nuPlan TTC**로 확장한다.


In [ ]:
def ttc_1d(gap, ego_speed, agent_speed):
    '''같은 차선 직선 주행을 가정한 고전적인 TTC [s].'''
    closing_speed = ego_speed - agent_speed

    # [TODO ①] 두 차량 사이가 가까워지지 않는 경우를 처리한다.
    # 힌트: closing speed의 부호가 거리 감소 여부를 결정한다.
    if ...:
        return np.inf

    # [TODO ②] 충돌까지 남은 시간을 계산한다.
    # 힌트: 거리 [m]와 상대속도 [m/s]를 어떻게 조합해야 결과가 초 [s]가 되는지 본다.
    return ...


# 8.4절에서 nuPlan projection 결과를 최종 0/1 점수로 바꿀 때 사용한다.
def ttc_gate(ttc_values, least_min_ttc=0.95):
    values = np.asarray(ttc_values, dtype=float)

    # [TODO ③] state sequence 전체에 strict bound를 적용한다.
    # 힌트: 값 하나가 경계와 같아도 실패하며, 모든 timestep이 통과해야 1점이다.
    return ...


try:
    approaching = ttc_1d(gap=20.0, ego_speed=10.0, agent_speed=5.0)  # 4 s
    separating = ttc_1d(gap=20.0, ego_speed=5.0, agent_speed=10.0)   # inf
    ok = np.isclose(approaching, 4.0) and np.isinf(separating)
    print("✅" if ok else "❌ TODO 확인", "TTC:", approaching, separating)
except Exception as e:
    print(f"❌ TODO를 채우십시오: {type(e).__name__}: {e}")


### 8.4 고전 TTC를 nuPlan 방식으로 확장하기

nuPlan은 `거리/상대속도`를 한 번 계산하는 대신, **ego와 주변 차량의 실제 크기를 가진 oriented box를
미래로 조금씩 움직여 처음 겹치는 시각**을 찾는다.

| 고전 TTC | nuPlan TTC |
|---|---|
| 같은 직선의 두 점 | heading과 크기를 가진 여러 oriented box |
| 앞차 하나 | 앞차와 교차 가능성이 있는 주변 객체 |
| 나눗셈 한 번 | `0.1, 0.2, ..., 2.9 s`의 이산 충돌 검사 |
| 1차원 gap | `in_collision(ego_box, track_box)` |

한 simulation timestep에서는 다음 순서로 계산한다.

1. ego 과실 충돌이 이미 발생했으면 `TTC=0`이다.
2. ego가 정지했거나 관련 객체가 없으면 `None`을 반환한다.
3. ego와 객체가 현재 속도와 heading을 유지한다고 가정한다.
4. oriented box를 `0.1 s`씩 이동하며 처음 겹치는 시각을 반환한다.
5. `3 s` 안에 겹치지 않으면 `None`을 반환한다.

이 계산을 state sequence의 모든 timestep에 반복한 뒤, TTC가 하나라도 `0.95 s` 이하이면 전체 metric은
0점이다.

$$
m_{TTC}=\mathbf 1\!\left[\forall t,\;TTC_t>0.95\right]
$$

아래에서는 nuPlan 내부 자료구조와 vectorized 좌표 갱신은 완성 코드로 제공한다. 실습자는 핵심인
두 oriented box의 충돌 여부만 채운다. synthetic scene을 원본 함수와 비교해 결과를 확인한다.


In [ ]:
from nuplan.common.actor_state.oriented_box import OrientedBox, in_collision
from nuplan.common.actor_state.state_representation import StateSE2
from nuplan.planning.metrics.evaluation_metrics.common.time_to_collision_within_bound import (
    _compute_time_to_collision_at_timestamp,
    _get_ego_tracks_displacement_info,
    _get_relevant_tracks,
)


def nuplan_ttc_at_timestamp(
    timestamp, ego_state, ego_speed,
    tracks_poses, tracks_speed, tracks_boxes,
    timestamps_at_fault_collisions=(),
    time_step_size=0.1, time_horizon=3.0,
    stopped_speed_threshold=5e-3,
):
    '''nuPlan _compute_time_to_collision_at_timestamp의 학습용 재구현.'''
    if timestamp in timestamps_at_fault_collisions:
        return 0.0
    if len(tracks_poses) == 0 or ego_speed <= stopped_speed_threshold:
        return None

    # 원본 입력을 함수 밖에서 바꾸지 않도록 복사한다.
    projected_tracks = np.asarray(tracks_poses, dtype=float).copy()
    tracks_speed = np.asarray(tracks_speed, dtype=float)
    info = _get_ego_tracks_displacement_info(
        ego_state, ego_speed, projected_tracks, tracks_speed, time_step_size)
    relevant = _get_relevant_tracks(
        info.ego_pose, info.ego_box, info.ego_dx, info.ego_dy,
        projected_tracks, tracks_boxes, info.tracks_dxy,
        time_step_size, time_horizon)
    if len(relevant) == 0:
        return None

    projected_ego_pose = info.ego_pose.copy()
    for tau in np.arange(time_step_size, time_horizon, time_step_size):
        # nuPlan helper가 계산한 한 step 변위를 현재 ego pose에 누적한다.
        projected_ego_pose[:2] += (info.ego_dx, info.ego_dy)
        projected_ego_box = OrientedBox.from_new_pose(
            info.ego_box, StateSE2(*projected_ego_pose))

        # [TODO] 모든 track의 한 step 변위를 vectorized 연산으로 누적한다.
        # 힌트: x·y만 갱신하고 heading은 유지한다. 변위는 info에 이미 계산되어 있다.
        projected_tracks[:, :2] = ...

        for idx in relevant:
            projected_track_box = OrientedBox.from_new_pose(
                tracks_boxes[idx], StateSE2(*projected_tracks[idx]))

            # [TODO] 현재 투영 시각에 두 oriented box가 충돌했는지 판정한다.
            # 힌트: 위에서 import한 collision predicate는 box 두 개를 받아 bool을 반환한다.
            collision = ...
            if collision:
                return float(tau)
    return None


# 실제 sample ego 앞 15 m에 더 느린 synthetic 차량을 놓는다.
h = sample_ego.center.heading
track_pose = np.array([
    sample_ego.center.x + 15.0 * np.cos(h),
    sample_ego.center.y + 15.0 * np.sin(h),
    h,
], dtype=float)
track_box = OrientedBox(StateSE2(*track_pose), length=4.5, width=2.0, height=1.5)
demo_args = dict(
    timestamp=sample_ego.time_point.time_us,
    ego_state=sample_ego,
    ego_speed=np.asarray(10.0),
    tracks_poses=np.asarray([track_pose]),
    tracks_speed=np.asarray([5.0]),
    tracks_boxes=np.asarray([track_box]),
    timestamps_at_fault_collisions=[],
    time_step_size=0.1,
    time_horizon=3.0,
    stopped_speed_threshold=5e-3,
)

try:
    mine_ttc_demo = nuplan_ttc_at_timestamp(**demo_args)
    reference_ttc_demo = _compute_time_to_collision_at_timestamp(**demo_args)
    ok = mine_ttc_demo is not None and np.isclose(mine_ttc_demo, reference_ttc_demo)
    print("✅" if ok else "❌ TODO 확인",
          "notebook:", mine_ttc_demo, "nuPlan:", reference_ttc_demo)
except Exception as e:
    print(f"❌ TODO를 채우십시오: {type(e).__name__}: {e}")

#### nuPlan TTC projection 시각화

왼쪽은 현재 box에서 시작해 등속으로 투영된 ego와 앞차를 보여준다. 투명한 box가 미래 위치이고,
붉은 테두리는 처음 충돌하는 위치다. 오른쪽은 state sequence에서 얻은 TTC 중 최솟값이 `0.95 s`를
넘어야만 1점이 되는 strict gate다. 고전식이 아니라 8.4절에서 구현한 nuPlan 결과를 시각화한다.
아래 셀도 제공 코드이며, 붉은 box가 나타난 시간과 오른쪽 gate의 경계 포함 여부만 확인한다.


In [ ]:
# 8.4 구현이 맞을 때만 oriented-box projection과 metric gate를 그린다.
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

try:
    if mine_ttc_demo is None or not np.isclose(mine_ttc_demo, reference_ttc_demo):
        raise ValueError("8.4 구현 결과가 nuPlan 원본과 아직 일치하지 않습니다.")

    dt = demo_args["time_step_size"]
    info = _get_ego_tracks_displacement_info(
        demo_args["ego_state"], demo_args["ego_speed"],
        demo_args["tracks_poses"].copy(), demo_args["tracks_speed"], dt,
    )
    collision_t = float(reference_ttc_demo)
    shown_times = sorted(set([0.0, 0.5, 1.0, 1.5, collision_t]))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    ax = axes[0]
    all_xy = []
    for tau in shown_times:
        steps = int(round(tau / dt))
        ego_pose = info.ego_pose.copy()
        ego_pose[:2] += steps * np.array([info.ego_dx, info.ego_dy])
        track_pose_t = demo_args["tracks_poses"][0].copy()
        track_pose_t[:2] += steps * info.tracks_dxy[0]

        ego_box = OrientedBox.from_new_pose(info.ego_box, StateSE2(*ego_pose))
        track_box_t = OrientedBox.from_new_pose(demo_args["tracks_boxes"][0], StateSE2(*track_pose_t))
        is_hit = np.isclose(tau, collision_t)
        for box, color in [(ego_box, "tab:blue"), (track_box_t, "tab:orange")]:
            xy = np.asarray(box.geometry.exterior.coords)[:, :2]
            all_xy.append(xy)
            ax.add_patch(MplPolygon(
                xy, closed=True, facecolor=color, alpha=.18 if tau else .45,
                edgecolor="red" if is_hit else color,
                linewidth=2.2 if is_hit else 1.0,
            ))
        ax.text(ego_pose[0], ego_pose[1] + 1.8, f"{tau:.1f}s", fontsize=8, ha="center")

    all_xy = np.concatenate(all_xy)
    margin = 4.0
    ax.set_xlim(all_xy[:, 0].min()-margin, all_xy[:, 0].max()+margin)
    ax.set_ylim(all_xy[:, 1].min()-margin, all_xy[:, 1].max()+margin)
    ax.set_aspect("equal"); ax.grid(alpha=.25)
    ax.set_title(f"Oriented-box projection — first collision {collision_t:.1f} s")
    ax.set_xlabel("global x [m]"); ax.set_ylabel("global y [m]")

    minimum_ttc = np.unique(np.r_[np.linspace(0.0, 3.0, 301), 0.95])
    gate_scores = [ttc_gate([value]) for value in minimum_ttc]
    axes[1].step(minimum_ttc, gate_scores, where="post", color="tab:green")
    axes[1].axvline(0.95, color="tab:red", ls="--", label="strict bound = 0.95 s")
    axes[1].scatter([0.95], [ttc_gate([0.95])], color="tab:red", zorder=3,
                    label="0.95 s also fails")
    axes[1].set(xlabel="minimum projected TTC [s]", ylabel="TTC metric score",
                ylim=(-0.05, 1.05), title="nuPlan TTC gate")
    axes[1].grid(alpha=.3); axes[1].legend(fontsize=8)
    fig.tight_layout(); plt.show()
except Exception as e:
    print(f"❌ 8.4 구현을 확인하십시오: {type(e).__name__}: {e}")


### 8절 구현 결과 정리

앞에서 Drivable area의 거리·누적 gate와 nuPlan TTC의 oriented-box projection까지 이미 구현했다.
여기서는 같은 로직을 다시 작성하지 않고 대표 입력의 결과만 한 표로 확인한다.

- Drivable area: FR 모서리가 `0.35 m` 이탈한 예제는 `0점`
- nuPlan TTC: synthetic 앞차의 최초 box 충돌 시각은 실습자 구현과 원본 모두 `2.1 s`
- TTC gate: 최솟값이 `0.95 s`와 같으면 실패하고, `0.95 s`보다 커야 통과

실제 주행 중에는 이 계산이 planning timestep마다 반복된다. 시간에 따라 누적되는 결과는 9절의
Closed-loop 실시간 막대·heatmap에서 확인한다.


In [ ]:
# 8.2와 8.4에서 이미 계산한 대표 결과만 모아 확인한다.
result_summary = pd.DataFrame([
    {
        "확인 항목": "Drivable area 누적 gate",
        "대표 입력": "FR corner outside distance = 0.35 m",
        "실습 결과": float(demo_score),
        "기대 결과": 0.0,
        "일치": bool(np.isclose(demo_score, 0.0)),
    },
    {
        "확인 항목": "nuPlan oriented-box TTC",
        "대표 입력": "ego 10 m/s, front agent 5 m/s",
        "실습 결과": float(mine_ttc_demo),
        "기대 결과": float(reference_ttc_demo),
        "일치": bool(np.isclose(mine_ttc_demo, reference_ttc_demo)),
    },
    {
        "확인 항목": "TTC strict gate",
        "대표 입력": "minimum TTC = 0.95 s",
        "실습 결과": float(ttc_gate([0.95])),
        "기대 결과": 0.0,
        "일치": bool(ttc_gate([0.95]) == 0.0),
    },
])
display(result_summary)
assert result_summary["일치"].all(), "앞 절의 구현 결과를 다시 확인하십시오."


## 9. nuPlan metric을 timestep마다 계산·누적해 보기

6~8절에서 확인한 nuPlan Closed-loop 지표를 실제 Non-reactive 주행에 연결한다. nuPlan은 보통
시나리오가 끝난 뒤 전체 simulation history로 metric을 계산하지만, 여기서는 같은 metric을 실제
상태가 한 step 추가될 때마다 다시 계산해 중간 과정을 화면에 보여준다. **지표 공식은 같고 계산·표시
시점만 다르다.**

### 9.1 한 timestep의 처리 흐름

| 순서 | 입력·계산 | 대시보드에 남기는 값 |
|---|---|---|
| 1 | 현재 실제 ego·observation 수집 | 실행 history에 한 step 추가 |
| 2 | 같은 nuPlan metric을 현재 timestep에 적용 | `step_*` 8개 |
| 3 | 시작부터 현재까지의 history prefix에 같은 metric 적용 | `cumulative_*` 8개 |
| 4 | 누적 8개 지표를 공식 곱셈·가중평균 식으로 집계 | `cumulative_final` |
| 5 | planner 후보 일부를 controller가 실행 | 다음 실제 상태로 이동 |

지표의 시간 의미는 원래 nuPlan 정의를 따른다. DAC·충돌·TTC는 현재 상태에서 판정할 수 있지만,
direction·comfort는 직전 시간 window가 필요하고 progress는 시작부터 현재까지의 진행량을 사용한다.
따라서 “매 timestep 계산”은 모든 지표를 pose 하나로 억지로 바꾼다는 뜻이 아니라, 그 시점까지 준비된
입력으로 같은 metric을 계산한다는 뜻이다.

> 충돌은 이벤트 지표다. `step_no_ego_at_fault_collisions=0`은 누적 gate가 그 step에서 처음
> `1→0`이 됐다는 뜻이다. 이후에는 새 충돌 이벤트가 없으면 step은 1이지만 cumulative는 계속 0이다.

> 누적값이 항상 내려가기만 하는 것은 아니다. DAC·충돌·TTC 같은 strict gate는 과거 실패가 남지만,
> speed compliance는 누적 overspeed를 경과시간으로 정규화하고 progress는 진행량 비율을 다시 계산하므로
> 이후 정상 주행이나 추격으로 회복될 수 있다. 강제로 `cummin`하면 nuPlan 원식과 달라진다.


### 9.2 실시간 집계 함수와 대시보드

`RealtimeNuPlanMetricTracker`가 실제 ego·observation을 한 step씩 받아 같은 nuPlan metric의
`step` 값과 `cumulative` 값을 반환한다. 지표별 입력 window를 맞추는 반복 코드는 helper가 담당한다.
실습자는 누적 지표 8개를 final 하나로 합치는 `realtime_metric_score()`를 완성한다.

대시보드는 다음 세 결과를 함께 보여준다.

- 오른쪽 아래: timestep마다 계산된 `step_*` 지표 기록
- 오른쪽 가운데: 현재까지의 `cumulative_*` 지표 8개
- 오른쪽 위: 누적 지표 8개로 계산한 `cumulative_final`

왼쪽 주행 프레임 안의 작은 표는 planner 미래 후보의 보조 진단이고, 오른쪽 세 패널은 실제 실행
history의 nuPlan metric이다. 두 값을 섞어 해석하지 않는다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MULTIPLICATIVE_NAMES = [
    "no_ego_at_fault_collisions", "drivable_area_compliance",
    "driving_direction_compliance", "ego_is_making_progress",
]
REALTIME_WEIGHTS = {
    "ego_progress_along_expert_route": 5.0,
    "time_to_collision_within_bound": 5.0,
    "speed_limit_compliance": 4.0,
    "ego_is_comfortable": 2.0,
}


def realtime_metric_score(metrics):
    """nuPlan의 곱셈 항 × 가중평균 식으로 누적 지표 8개를 집계한다."""
    values = dict(metrics)

    # [TODO ①] 네 곱셈 지표를 하나의 gate로 결합한다.
    # 힌트: 하나가 0이면 gate도 0이어야 한다.
    gate = ...

    # [TODO ②] 네 가중 지표의 가중평균을 계산한다.
    # 힌트: 가중치는 5·5·4·2이고 가중치 합으로 정규화한다.
    weighted_average = ...

    return float(gate * weighted_average)


def check_realtime_metric_score(fn):
    perfect = {name: 1.0 for name in CLOSED_BREAKDOWN}
    cases = [
        (perfect, 1.0),
        (dict(perfect, drivable_area_compliance=0.0), 0.0),
        (dict(perfect, speed_limit_compliance=0.5), 14.0 / 16.0),
        (dict(perfect, ego_progress_along_expert_route=0.0), 11.0 / 16.0),
        (dict(perfect, ego_is_making_progress=0.0), 0.0),
    ]
    try:
        ok = all(np.isclose(fn(metrics), expected) for metrics, expected in cases)
        print("✅ 실시간 집계 함수 완성" if ok else "❌ TODO의 집계식을 확인하십시오.")
    except Exception as e:
        print(f"❌ TODO를 채우십시오: {type(e).__name__}: {e}")


check_realtime_metric_score(realtime_metric_score)


def plot_live_closed_loop_dashboard(frame, rows, dt=0.1):
    """실제 주행, timestep 지표, 누적 지표와 누적 final을 함께 그린다."""
    rows = rows.sort_values("iteration").reset_index(drop=True)
    step_cols = [f"step_{name}" for name in CLOSED_BREAKDOWN]
    cumulative_cols = [f"cumulative_{name}" for name in CLOSED_BREAKDOWN]
    missing = [column for column in step_cols + cumulative_cols if column not in rows]
    if missing:
        raise KeyError(f"실시간 또는 누적 지표 열이 없습니다: {missing}")

    time_s = rows["iteration"].to_numpy(dtype=float) * dt
    step_values = rows[step_cols].to_numpy(dtype=float)
    cumulative_values = rows[cumulative_cols].to_numpy(dtype=float)
    latest_cumulative = cumulative_values[-1]
    cumulative_final = np.array([
        realtime_metric_score({
            name: row[f"cumulative_{name}"] for name in CLOSED_BREAKDOWN
        })
        for _, row in rows.iterrows()
    ])

    fig = plt.figure(figsize=(18, 10))
    grid = fig.add_gridspec(3, 2, width_ratios=[1.45, 1.0], height_ratios=[1, 1, 1.4])
    ax_video = fig.add_subplot(grid[:, 0])
    ax_final = fig.add_subplot(grid[0, 1])
    ax_cumulative = fig.add_subplot(grid[1, 1])
    ax_timeline = fig.add_subplot(grid[2, 1])

    ax_video.imshow(frame)
    ax_video.set_title(f"Executed Closed-loop — iter {int(rows.iloc[-1]['iteration'])}")
    ax_video.axis("off")

    ax_final.plot(time_s, cumulative_final, "o-", lw=1.8, ms=3)
    ax_final.set(ylabel="cumulative final", ylim=(-0.05, 1.05),
                 title="Cumulative score through each timestep")
    ax_final.grid(alpha=.3)

    names = list(CLOSED_BREAKDOWN)
    colors = ["tab:green" if value >= .999 else
              ("tab:red" if value <= .001 else "tab:orange")
              for value in latest_cumulative]
    ax_cumulative.barh(names, latest_cumulative, color=colors)
    ax_cumulative.set(xlim=(0, 1.05), title="Cumulative metrics through current timestep")
    ax_cumulative.grid(axis="x", alpha=.3)
    ax_cumulative.tick_params(axis="y", labelsize=8)

    time_edges = np.concatenate([time_s, [time_s[-1] + dt]])
    metric_edges = np.arange(len(names) + 1)
    image = ax_timeline.pcolormesh(
        time_edges, metric_edges, step_values.T, shading="flat",
        vmin=0, vmax=1, cmap="RdYlGn",
    )
    ax_timeline.set_xlim(time_edges[0], time_edges[-1])
    ax_timeline.set_ylim(len(names), 0)
    ax_timeline.set_yticks(np.arange(len(names)) + .5)
    ax_timeline.set_yticklabels(names, fontsize=8)
    ax_timeline.set(xlabel="planning time [s]  (each cell: [t, t + dt))",
                    title="nuPlan metrics computed at each timestep")
    fig.colorbar(image, ax=ax_timeline, label="metric value", pad=.01)
    fig.tight_layout()
    return fig


### 9.3 PLUTO Non-reactive Closed-loop 준비

샘플 시나리오를 iteration 0부터 다시 시작한다. `make_sim(scenario)`의 기본 controller는
`two_stage_controller`, observation은 `TracksObservation`이므로 Non-reactive Closed-loop다.
`LIVE_STEPS=31`이면 약 3초 동안 실제 상태와 지표 변화를 확인한다.


In [ ]:
from pathlib import Path
from src.planners.evaluator.realtime_metric_tracker import RealtimeNuPlanMetricTracker
import time
import pandas as pd
from IPython.display import Video, clear_output, display

live_dir = REPO_ROOT / "practice/results/live_closed_loop"
live_video_dir = live_dir / f"video_{time.time_ns()}"
live_dir.mkdir(parents=True, exist_ok=True)
live_video_dir.mkdir(parents=True, exist_ok=True)
live_planner = RefinementPlanner(
    model_adapter=adapter,
    scenario=scenario,
    use_refinement=False,
    driving_policy="ml",
    render=True,
    log_csv=False,         # 아래 tracker가 직접 계산하므로 중복 계산은 끈다.
    render_mode="closed",
    score_open_loop=False,
    save_dir=str(live_video_dir),
)
live_sim = make_sim(scenario)
live_planner.initialize(live_sim.initialize())
# 실제 실행 history에 nuPlan metric을 step별·누적으로 적용한다.
live_metric_tracker = RealtimeNuPlanMetricTracker(scenario)
live_metric_rows = []

print("Closed-loop 준비 완료:", scenario.token)


### 9.4 실제 주행과 step·누적 지표 함께 보기

아래 heatmap의 `step_*`에서 현재 순간을 먼저 보고, 가운데 `cumulative_*`와 위
`cumulative_final`에서 그 순간까지의 영향을 확인한다.

왼쪽 렌더 하단의 작은 표는 planner가 낸 미래 후보 전체의 점수다. 오른쪽은 실제로 실행된 ego와
observation에 같은 nuPlan metric을 적용한 결과다. 예를 들어 현재 ego가 Drivable area 안에 있고 미래
후보의 `+7.8 s` 지점만 벗어났다면, 왼쪽 후보 DAC는 0이어도 오른쪽 step·누적 DAC는 1이다.

누적 final이 올라가는 경우도 오류는 아니다. speed compliance와 progress는 nuPlan 수식상 이후 정상
주행으로 회복될 수 있고, DAC·충돌·TTC 같은 strict gate는 과거 실패가 남아 복구되지 않는다.


In [ ]:
LIVE_STEPS = 31

t0 = time.time()
for _ in range(LIVE_STEPS):
    if not live_sim.is_simulation_running():
        break

    planner_input = live_sim.get_planner_input()
    iteration = int(planner_input.iteration.index)
    current_ego, current_observation = planner_input.history.current_state

    # 미래 후보와 왼쪽 주행 프레임을 만든다.
    trajectory = live_planner.compute_trajectory(planner_input)

    # 실제 실행 상태 한 step에 같은 nuPlan metric을 적용한다.
    step_metrics, cumulative_metrics = live_metric_tracker.update(
        ego_state=current_ego,
        observation=current_observation,
        expert_ego_state=scenario.get_ego_state_at_iteration(iteration),
        baseline_path=live_planner._safe_baseline_path(current_ego),
    )
    live_metric_rows.append({
        "iteration": iteration,
        **{f"step_{name}": value for name, value in step_metrics.items()},
        **{f"cumulative_{name}": value for name, value in cumulative_metrics.items()},
        "cumulative_final": realtime_metric_score(cumulative_metrics),
    })

    live_sim.propagate(trajectory)

    live_rows = pd.DataFrame(live_metric_rows)
    current_frame = live_planner._imgs[-1]
    clear_output(wait=True)
    fig = plot_live_closed_loop_dashboard(
        current_frame, live_rows, dt=float(scenario.database_interval))
    display(fig)
    plt.close(fig)

elapsed = time.time() - t0
print(f"{len(live_rows)} timestep 완료 | {elapsed:.1f} s")
display(live_rows.tail(5)[[
    "iteration", "cumulative_final",
    "step_drivable_area_compliance", "cumulative_drivable_area_compliance",
    "step_ego_progress_along_expert_route", "cumulative_ego_progress_along_expert_route",
]].round(4))

live_planner.generate_planner_report()
live_video = live_video_dir / f"{scenario.log_name}_{scenario.token}.mp4"
print("video:", live_video)
display(Video(str(live_video), embed=True, width=900))


## 10. 모델 실행과 executed-history·공식 결과 분석

앞 단원에서는 한 시나리오의 앞 31 timestep을 직접 돌리며 실제 실행 history의 순간·누적 지표가
갱신되는 과정을 확인했다. 이번 단원은 같은 기록을 두 모델의 전체 시나리오 실행으로 확장하고 세 자료를
연결한다.

- timestep CSV의 `step_*`: simulator에서 실제 실행된 **그 순간**의 지표
- timestep CSV의 `cumulative_*`: 시작부터 현재 iteration까지 실행된 history prefix의 지표
- 영상: 위 state·observation이 만들어진 실제 Closed-loop 주행
- 공식 parquet: 시나리오 종료 후 nuPlan evaluator가 전체 history로 계산한 최종 결과

`ml_*` 열은 planner가 그 iteration에 제안한 미래 후보 궤적의 진단값이다. 실제로 실행된 history가
아니므로 10.2의 실패 timeline에는 사용하지 않는다.

### 10.1 PLUTO와 Diffusion Non-reactive 실행 — 시나리오 수 제한

주 비교 조건인 Non-reactive에서 두 모델을 먼저 실행하고 CSV·영상·공식 점수를 끝까지 분석한다.
두 실행은 adapter만 다르고 `use_refinement=false`이므로 raw 모델 궤적을 그대로 주행한다.
기본값은 `practice_scenarios` 중 앞의 **2개 시나리오**다. 빠른 동작 확인만 할 때는
`NONREACTIVE_SCENARIO_LIMIT=1`로 바꿀 수 있으며, 두 모델에는 항상 같은 제한이 적용된다. UID에도
제한 개수가 들어가므로 1개 결과와 2개 결과가 캐시에서 섞이지 않는다. Hydra 설정의
adapter class도 확인하므로 PLUTO와 Diffusion 결과가 뒤바뀌면 즉시 알 수 있다.
Reactive 평가는 주 분석이 끝난 10.4절에서 별도로 실행한다.


In [ ]:
# 주 비교: 같은 시나리오 수·같은 non-reactive 프리셋, adapter만 다르다.
NONREACTIVE_SCENARIO_LIMIT = 2
# executed-history step_* / cumulative_* 열을 포함하는 새 결과 스키마다.
NONREACTIVE_UID = f"practice2_{NONREACTIVE_SCENARIO_LIMIT}scenarios_executed_metrics_v2"

CLOSED = run_sim(
    "closed_loop_nonreactive_agents",
    uid=f"{NONREACTIVE_UID}/pluto", limit=NONREACTIVE_SCENARIO_LIMIT,
    video_dir="videos",
)
DIFF_CLOSED = run_sim(
    "closed_loop_nonreactive_agents",
    uid=f"{NONREACTIVE_UID}/diffusion", limit=NONREACTIVE_SCENARIO_LIMIT,
    adapter="diffusion", video_dir="videos",
)
print("PLUTO     :", CLOSED)
print("Diffusion :", DIFF_CLOSED)

### 10.2 실제 실행 history의 timestep·누적 지표와 공식 최종 결과

CSV 한 행은 `compute_trajectory()`가 호출된 planning iteration 하나다. CSV를 만들 때 planner는
현재 simulation history의 마지막 ego state와 observation을 tracker에 추가하고 다음 값을 기록한다.

- `step_<metric>`: 현재 iteration의 값. 충돌은 그 순간 새로 발생한 과실 충돌 이벤트를 뜻한다.
- `cumulative_<metric>`: iteration 0부터 현재까지의 history prefix로 다시 계산한 누적 지표
- `step_final`, `cumulative_final`: 각각의 8개 지표를 곱셈·가중평균 식으로 모은 온라인 점수

이 값들은 후보 미래 궤적의 `ml_*`와 다르다. 예를 들어 후보가 7초 뒤 차선을 벗어나더라도 현재
실행 pose가 영역 안이면 `step_drivable_area_compliance=1`이다. 이후 실제 ego가 영역을 벗어난
iteration에만 실제 지표가 낮아진다.

공식 evaluator는 시나리오가 종료된 뒤 전체 history를 한 번 평가한다. 따라서 주행 중 생성되는 CSV에는
아직 공식 최종 점수가 존재하지 않는다. 아래 셀은 실행이 끝난 뒤 parquet의 `official_final`과
`official_collision`을 같은 model·token으로 결합한다. 이 두 official 열은 10.3에서도 **같은
`OFFICIAL_TABLES` 객체**를 사용하므로 두 절의 최종 결과가 달라질 수 없다.

위 그래프는 매 iteration의 `cumulative_final`, 가운데는 `step_*`, 아래는 `cumulative_*`다.
붉은 선은 실제 누적 곱셈 지표가 처음 실패한 시점이며, 그런 실패가 없으면 누적 점수의 가장 큰 하락
시점을 표시한다. iteration은 0부터 시작하는 index이고 이 실행에서는 한 step이 0.1초다.


In [ ]:
MODEL_RUNS = {"PLUTO": CLOSED, "Diffusion": DIFF_CLOSED}
EXPECTED_ADAPTER = {
    "PLUTO": "PlutoModelAdapter",
    "Diffusion": "DiffusionModelAdapter",
}
SIMULATION_DT_S = 0.1

# CSV schema·adapter·token 결합을 검증하고 official 표를 한 번만 만든다.
STEP_RUNS, OFFICIAL_TABLES = load_executed_metric_runs(
    MODEL_RUNS, EXPECTED_ADAPTER
)


#### 실패 지점 선택과 timeline 그리기

CSV 병합, 첫 gate 실패 탐색, 3단 그래프처럼 반복적인 코드는 helper에 두었다. 여기서는 세 함수의
역할과 반환값만 읽으면 된다.

1. `load_executed_metric_runs()`: 실제 `step_*`·`cumulative_*` 열과 official 결과를 token으로 결합
2. `summarize_executed_metric_runs()`: 모델·시나리오별 대표 저하 시점과 공통 token 선택
3. `show_executed_metric_analysis()`: 대표 시점 전후 표와 전체 timeline 출력

`OFFICIAL_TABLES`는 다음 영상 셀에서도 그대로 재사용한다.


In [ ]:
executed_alerts, common_tokens, STEP_TOKEN = summarize_executed_metric_runs(
    STEP_RUNS, dt_s=SIMULATION_DT_S
)
display(executed_alerts)
print("상세 timeline과 두 모델 영상을 연결할 token:", STEP_TOKEN)

show_executed_metric_analysis(
    STEP_RUNS, executed_alerts, STEP_TOKEN, dt_s=SIMULATION_DT_S
)


### 10.3 영상과 공식 시나리오 점수

10.2의 heatmap은 실제 ego state·observation으로 계산한 `step_*`와 `cumulative_*`다. 아래 parquet
표는 같은 실행 history를 시나리오 종료 후 공식 evaluator가 집계한 결과다. 두 절의
`official_final`과 `official_collision`은 같은 `OFFICIAL_TABLES`에서 읽으므로 model·token이
같으면 반드시 동일하다. 온라인 `cumulative_final`은 매 prefix에서 확인하는 중간값이므로
시나리오 종료 후의 `official_final`과 이름과 역할을 구분한다.

10.1에서 실행한 공통 시나리오를 모두 펼치고, 각 시나리오마다 PLUTO와 Diffusion 영상을 나란히
배치한다. 10.2에서 선택된 붉은 선의 iteration을 영상 시간(`iteration × 0.1 s`)으로 옮겨 다음을
확인한다.

1. `step_*`에서 그 순간 낮아진 지표와 실제 장면을 비교한다.
2. `cumulative_*`에서 과거 위반이 남거나 이후 회복되는 과정을 확인한다.
3. 종료 후 official 지표가 실제 시나리오 실패로 판정했는지 확인한다.

공식 `no_ego_at_fault_collisions`는 모든 접촉을 동일하게 처리하지 않는다. `1.0`은 ego 과실 충돌
없음, `0.5`는 설정상 일부 충돌의 부분 점수, `0.0`은 최종 점수를 0으로 만드는 곱셈 gate다.
따라서 충돌처럼 보이는 장면은 영상만으로 단정하지 말고 이 열과 `official_final`을 함께 읽는다.

Non-reactive와 Reactive 결과는 서로 다른 표로 유지한다.


In [ ]:
from IPython.display import HTML

VIDEO_RUNS = {"PLUTO": CLOSED, "Diffusion": DIFF_CLOSED}
VIDEO_TAGS = {"PLUTO": "closed_pluto", "Diffusion": "closed_diffusion"}
VIDEO_COLORS = {"PLUTO": "#7b2cbf", "Diffusion": "#1976d2"}

# 브라우저에서 하나씩 고르는 대신, 이번에 실행한 공통 시나리오를 모두 펼친다.
video_files = {
    model: collect_videos(run_dir, VIDEO_TAGS[model])
    for model, run_dir in VIDEO_RUNS.items()
}
# 10.2의 official_final과 같은 객체를 재사용하므로 두 절의 최종값은 항상 같다.
score_tables = OFFICIAL_TABLES

for scenario_number, token in enumerate(common_tokens, start=1):
    section(f"Scenario {scenario_number}/{len(common_tokens)} | token={token}")
    cards, score_rows = [], []

    for model_name in ("PLUTO", "Diffusion"):
        row = score_tables[model_name].loc[token]
        collision_score = float(row["no_ego_at_fault_collisions"])
        official_final = float(row["score"])
        if np.isclose(collision_score, 0.0) and not np.isclose(official_final, 0.0):
            raise AssertionError("공식 collision=0인데 official final이 0이 아닙니다.")
        path = video_files[model_name].get(row["video"])
        heading = (f"<h3 style='margin:0 0 8px;color:{VIDEO_COLORS[model_name]}'>"
                   f"{model_name} · official final {official_final:.3f} "
                   f"· collision {collision_score:.3f}</h3>")
        if path is None:
            body = "<p>저장된 영상이 없습니다. runner 상태를 확인하십시오.</p>"
        else:
            relative_path = f"{VIDEO_STAGE}/{VIDEO_TAGS[model_name]}/{path.name}"
            body = video_html(relative_path, width=590)
        cards.append(f"<div style='flex:1;min-width:500px'>{heading}{body}</div>")

        score_rows.append({
            "model": model_name,
            "official_final": official_final,
            **{name: float(row[name]) for name in CLOSED_BREAKDOWN},
        })

    display(HTML("<div style='display:flex;gap:18px;flex-wrap:wrap'>" +
                 "".join(cards) + "</div>"))
    display(pd.DataFrame(score_rows).set_index("model").round(3))


#### Non-reactive 전체 지표 비교

영상에서는 시나리오별 차이를 확인했다. 이제 공통 시나리오 평균으로 어떤 지표에서 모델 차이가
생겼는지 본다. 표의 `score` 행이 최종 점수 평균이고, `×` 표시는 곱셈 항이다.


In [ ]:
# 같은 Non-reactive 조건의 공통 시나리오만 평균낸다.
RUNS = {"PLUTO": CLOSED, "Diffusion": DIFF_CLOSED}
breakdown = compare_breakdown(RUNS)
display(breakdown.round(3))
plot_breakdown_comparison(breakdown, title="Closed-loop 세부 지표 — Non-reactive")
plt.show()


### 10.4 Reactive는 별도 조건으로 비교

이제 observation만 IDM으로 바꾸어 두 모델을 실행한다. Reactive는 ego에 반응하는 주변 차량까지 함께
달라지는 별도 simulation이므로 Non-reactive 점수와 직접 빼지 않는다. 실행 시간이 부족하면 이 절은
선택적으로 진행해도 된다.


In [ ]:
# 보조 비교: observation만 reactive(IDM)로 바꾼다. 두 모델 모두 실행해야 한다.
REACTIVE_SCENARIO_LIMIT = NONREACTIVE_SCENARIO_LIMIT
REACTIVE_UID = f"practice2_{REACTIVE_SCENARIO_LIMIT}scenarios_reactive"
REACTIVE = run_sim(
    "closed_loop_reactive_agents", uid=f"{REACTIVE_UID}/pluto",
    limit=REACTIVE_SCENARIO_LIMIT, video_dir="videos"
)
DIFF_REACTIVE = run_sim(
    "closed_loop_reactive_agents", uid=f"{REACTIVE_UID}/diffusion",
    limit=REACTIVE_SCENARIO_LIMIT, adapter="diffusion", video_dir="videos",
)
print("PLUTO reactive     :", REACTIVE)
print("Diffusion reactive :", DIFF_REACTIVE)

In [ ]:
# Reactive는 별도 평가다. 프리셋을 가로지르는 차이를 모델 성능 차이로 해석하지 않는다.
REACTIVE_RUNS = {"PLUTO": REACTIVE, "Diffusion": DIFF_REACTIVE}
for name, run in REACTIVE_RUNS.items():
    print(f"{name:10s} reactive 공식 최종 점수 {final_score(run):.4f}")
display(compare_scenario_scores(REACTIVE_RUNS))

## 11. Open-loop과 Closed-loop, 그리고 궤적 후처리

### 11.1 모델별 Open-loop·Closed-loop 대조

두 점수는 모두 0~1이지만 같은 성능을 재지 않는다.

- Open-loop: 로그 궤적을 얼마나 잘 모사했는가
- Closed-loop: 계획을 반복 실행했을 때 안전하고 진행 가능한가

3~5절의 `OPEN`·`DIFF_OPEN`과 10절의 `CLOSED`·`DIFF_CLOSED` 결과를 재사용한다. 점수 차이를 같은
성능 척도의 향상·하락으로 해석하면 안 된다. 다만 `open_loop - closed_loop`를 **mismatch 탐색용
screening 값**으로 사용하면, 로그 모사는 잘하지만 실제 실행 점수가 낮은 시나리오를 빠르게 찾을 수 있다.

아래 셀은 screening gap이 큰 순서로 token을 출력하고 상위 사례를 산점도에 표시한다. 그 token을
10절 영상 브라우저에서 다시 열어 어떤 지표가 Closed-loop 실패를 만들었는지 확인한다.


In [ ]:
import pandas as pd

# 3~5절과 10절에서 만든 공식 평가 결과를 그대로 사용한다.
OPEN_CLOSED_RUNS = {
    "PLUTO": (OPEN, CLOSED),
    "Diffusion": (DIFF_OPEN, DIFF_CLOSED),
}

paired = []
for model_name, (open_run, closed_run) in OPEN_CLOSED_RUNS.items():
    open_part = per_scenario_scores(open_run)[["token", "scenario_type", "score"]].rename(
        columns={"score": "open_loop"}
    )
    closed_part = per_scenario_scores(closed_run)[["token", "score"]].rename(
        columns={"score": "closed_loop"}
    )
    both = open_part.merge(closed_part, on="token", validate="one_to_one")
    both.insert(0, "model", model_name)
    paired.append(both)

open_closed = pd.concat(paired, ignore_index=True)
open_closed["screening_gap"] = open_closed["open_loop"] - open_closed["closed_loop"]
mismatch = open_closed.sort_values("screening_gap", ascending=False).reset_index(drop=True)
display(mismatch[["model", "token", "scenario_type", "open_loop",
                  "closed_loop", "screening_gap"]].round(3))

# screening gap 상위 세 사례를 그림에 표시한다. gap은 공통 성능 단위가 아니라 탐색 순서일 뿐이다.
top_mismatch = mismatch.head(min(3, len(mismatch)))
fig, ax = plt.subplots(figsize=(7, 5))
for model_name, group in open_closed.groupby("model"):
    ax.scatter(group["open_loop"], group["closed_loop"], s=65, alpha=0.8, label=model_name)
ax.set_xlabel("Open-loop score — 로그 궤적 모사")
ax.set_ylabel("Closed-loop score — 실제 주행 안전·품질")
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
for row in top_mismatch.itertuples():
    ax.annotate(f"{row.model}:{row.token[:8]}",
                (row.open_loop, row.closed_loop), xytext=(5, 5),
                textcoords="offset points", fontsize=8)
ax.grid(alpha=0.3); ax.legend(); ax.set_title("같은 모델·시나리오의 서로 다른 두 질문")
plt.show()

### 11.2 왜 궤적 후처리가 필요한가

Open-loop 오차가 작아도 실행 단계에서는 다음 문제가 나타날 수 있다.

| raw ML 궤적의 문제 | Closed-loop에서 보이는 신호 | 후처리 방향 |
|---|---|---|
| 짧은 차선 이탈·위험한 곡률 | Drivable area 또는 TTC gate가 0 | 지도·충돌 제약 적용 |
| waypoint의 작은 흔들림 | 가속도·jerk와 comfort 악화 | 저역통과 평활화 |
| controller가 계획을 정확히 못 따라감 | 계획 궤적과 실제 실행 궤적의 간격, 다음 상태 오차 누적 | 동역학을 고려한 최적화 |

후처리는 Open-loop 점수를 꾸미는 단계가 아니라 raw 궤적을 **추종 가능하고 제약을 만족하는 궤적**으로
바꾸는 단계다. 실습 3에서는 동일한 CSV·heatmap·영상으로 raw ML, smoothing, MPC를 비교한다.

## 12. 정리

이번 실습의 분석 흐름은 다음 한 줄로 이어진다.

```text
평가 조건 고정 → 모델별 Open-loop 비교 → 개별 지표 구현·시각화
→ 실제 Closed-loop 주행과 실시간 지표 관찰 → 전체 결과 분석 → 후처리 필요성 도출
```

기억할 점은 네 가지다.

1. Non-reactive와 Reactive는 observation이 다르므로 같은 프리셋 안에서 모델을 비교한다.
2. `step_*`는 현재 순간, `cumulative_*`는 현재까지의 실행 history, official 점수는 완료된 시나리오를 본다.
3. 곱셈 항은 한 번의 안전 위반으로 전체 점수를 0으로 만들 수 있다.
4. 높은 Open-loop 점수는 Closed-loop 안전을 보장하지 않으므로 궤적 후처리를 별도로 평가해야 한다.

다음 문서 [practice3_ml_planner_refinement.ipynb](practice3_ml_planner_refinement.ipynb)에서 저역통과 필터와
MPC를 적용하고, 후처리 전후의 timestep CSV·영상·공식 Closed-loop 지표를 비교한다.